In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
app_train = pd.read_csv("../Data/application_train.csv").reset_index(drop=True)
app_test = pd.read_csv("../Data/application_test.csv").reset_index(drop=True)

In [ ]:
app_train.head()

In [ ]:
app_train.describe()

In [ ]:
# KIỂM TRA TỈ LỆ VỠ NỢ (1 - VỠ NỢ)
counts = app_train["TARGET"].value_counts()
total = len(app_train)
ratio = counts / total * 100
result = pd.DataFrame(
    {"target": counts.index, "số lượng": counts.values, "tỉ lệ": ratio.values}
)
print("Tổng số lượng: ", total)
print(result)

### DỮ LIỆU LỆCH NGHIÊM TRỌNG

In [ ]:
app_train.select_dtypes("object").head()

In [ ]:
# KIỂM TRA GIÁ TRỊ THIẾU


def check_missing_values(df):
    missing_values = df.isnull().sum()
    total = len(df)
    ratio = round(missing_values / total * 100, 2)
    result = pd.DataFrame(
        {
            "cột": missing_values.index,
            "số lượng": missing_values.values,
            "tỉ lệ": ratio.values,
        }
    )
    print("Dữ liệu có: " + str(len(df.columns)) + " cột")
    print("Số cột có giá trị thiếu: " + str((missing_values > 0).sum()) + " cột")
    return result


check_missing_values(app_train).sort_values(by="tỉ lệ", ascending=False)

### MISSING VALUE RẤT NHIỀU

In [ ]:
check_missing_values(app_test).sort_values(by="tỉ lệ", ascending=False)

In [ ]:
# VẼ BIỂU ĐỒ XEM PHÂN BỐ
# KIỂM TRA SỐ LƯỢNG CÁC LOẠI DỮ LIỆU

print(app_train.dtypes.value_counts())
print("=" * 50)
print(app_train.select_dtypes("object").apply(lambda x: x.nunique()))

# 1. Tạo danh sách cột object (bỏ qua 1 số cột có quá nhiều giá trị duy nhất)

dtypes_object = app_train.select_dtypes(include=["object"]).columns.tolist()
excluded_columns = ["OCCUPATION_TYPE", "ORGANIZATION_TYPE"]
dtypes_object = [col for col in dtypes_object if col not in excluded_columns]


# 2. Vẽ biểu đồ phân bố cho các cột object còn lại
def plot_stats(feature, label_rotation=False, horizontal_layout=True):
    """
    Vẽ 2 biểu đồ cạnh nhau (hoặc chồng dọc):
    - Bên trái: số lượng hợp đồng theo từng category
    - Bên phải: tỷ lệ % TARGET=1 theo từng category
    """
    # Số lượng khoản vay mỗi category
    temp = app_train[feature].value_counts()
    df1 = pd.DataFrame({feature: temp.index, "Số lượng": temp.values})
    # Tỷ lệ % TARGET=1 mỗi category
    cat_perc = app_train.groupby(feature, as_index=False).agg(
        default_rate=("TARGET", "mean")
    )
    cat_perc = cat_perc.sort_values(by="default_rate", ascending=False)

    if horizontal_layout:
        fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12, 6))
    else:
        fig, (ax1, ax2) = plt.subplots(nrows=2, figsize=(12, 14))

    sns.set_color_codes("pastel")

    # Biểu đồ số lượng bên trái
    sns.barplot(ax=ax1, x=feature, y="Số lượng", data=df1)
    if label_rotation:
        ax1.set_xticklabels(ax1.get_xticklabels(), rotation=90)
    ax1.set_title(f"Khoản vay theo {feature}", fontsize=12)

    # Biểu đồ tỷ lệ vỡ nợ bên phải
    sns.barplot(
        ax=ax2, x=feature, y="default_rate", order=cat_perc[feature], data=cat_perc
    )
    if label_rotation:
        ax2.set_xticklabels(ax2.get_xticklabels(), rotation=90)
    ax2.set_title(f"Tỷ lệ vỡ nợ theo {feature}", fontsize=12)
    ax2.set_ylabel("Tỷ lệ vỡ nợ (%)", fontsize=10)

    plt.tight_layout()
    plt.show()


# 3. Vẽ biểu đồ cho các cột object còn lại
for col in dtypes_object:
    print(f"Biểu đồ phân bố cho cột: {col}")
    plot_stats(col, label_rotation=True, horizontal_layout=True)

# Insights từ phân tích biến Categorical (Object Features)

## 1. Nhóm Signal Mạnh — Ưu tiên giữ cho model

### NAME_EDUCATION_TYPE
- Xu hướng đơn điệu rõ ràng: học vấn càng thấp → default rate càng cao.
- `Lower secondary` ~12% vs `Academic degree` ~2% — chênh lệch 6x.
- Đây là feature categorical có signal mạnh nhất trong toàn bộ nhóm object.
- Phần lớn mẫu tập trung ở `Secondary/secondary special` (~220k), học vấn trung bình.

### NAME_INCOME_TYPE
- `Maternity leave` (~40%) và `Unemployed` (~36%) có default rate cực cao — nhưng sample rất nhỏ (<1k mỗi nhóm), cần thận trọng khi diễn giải.
- `Working`, `Commercial associate`, `Pensioner`, `State servant` có gradient rõ từ ~10% xuống ~6%.
- Gợi ý: cân nhắc gộp `Maternity leave` + `Unemployed` thành 1 nhóm "Không có thu nhập ổn định" để tránh overfitting do sample nhỏ.

### NAME_HOUSING_TYPE
- `Rented apartment` (~12.5%) và `With parents` (~11.7%) có default rate cao nhất — phản ánh thiếu tài sản và sự phụ thuộc.
- `House/apartment` (~8%) thấp hơn đáng kể dù chiếm ~270k mẫu (áp đảo).
- Office apartment có default rate thấp nhất (~6.5%) — nhóm ổn định nhưng rất ít mẫu.

### CODE_GENDER
- Nam (~10%) vỡ nợ nhiều hơn Nữ (~7%) một cách nhất quán.
- `XNA` gần như 0% default nhưng sample cực nhỏ — nên xử lý như missing value hoặc gộp vào nhóm khác.

---

## 2. Nhóm Signal Trung Bình — Giữ nhưng cân nhắc

### NAME_CONTRACT_TYPE
- `Cash loans` (~8.2%) rủi ro hơn `Revolving loans` (~5.5%).
- Chỉ 2 giá trị → encode đơn giản bằng binary (0/1).

### NAME_FAMILY_STATUS
- `Civil marriage` (~10%) và `Single/not married` (~9.8%) rủi ro cao hơn `Married` (~7.5%) và `Widow` (~5.8%).
- Gradient có nhưng không quá mạnh. `Unknown` gần như 0% — sample quá nhỏ, nên gộp hoặc loại.

### WALLSMATERIAL_MODE
- `Wooden` (~9.5%) cao nhất, `Monolithic` (~5%) thấp nhất — chênh lệch ~2x.
- Tuy nhiên hầu hết vật liệu hiếm có sample rất nhỏ (<5k), dễ gây noise.
- `Panel` và `Stone, brick` chiếm phần lớn mẫu với default rate trung bình (~7-8%).

### FLAG_OWN_CAR
- Không có xe (~8.5%) vs có xe (~7.5%) — chênh lệch nhỏ (~1%).
- Signal yếu nhưng có hướng hợp lý (có tài sản → ít rủi ro hơn).

---

## 3. Nhóm Signal Yếu — Cân nhắc loại bỏ

### WEEKDAY_APPR_PROCESS_START
- Default rate dao động rất nhỏ giữa các ngày (~7.8% đến ~8.3%).
- Phân bố số lượng hợp đồng thấp hơn vào cuối tuần (Saturday, Sunday) — nhưng không ảnh hưởng đến default rate.
- Kết luận: Nhiều khả năng là noise, không nên đưa vào model.

### FLAG_OWN_REALTY
- Chênh lệch gần như không đáng kể: có nhà (~8%) vs không có (~8.3%).
- Signal quá yếu để có giá trị dự báo.

### NAME_TYPE_SUITE
- `Unaccompanied` chiếm ~250k / ~315k tổng mẫu — quá dominant.
- Các nhóm còn lại sample nhỏ, default rate chênh nhau không nhiều (~7.5% đến ~9.7%).
- Signal yếu, phân bố mất cân bằng nghiêm trọng.

### FONDKAPREMONT_MODE
- `reg oper account` chiếm ~73k trong tổng ~96k mẫu có giá trị.
- Default rate chênh nhau nhỏ (~5.8% đến ~7.5%).
- Thêm vào đó feature này có missing value rất nhiều (chỉ ~96k / 307k có giá trị).

### HOUSETYPE_MODE
- `block of flats` chiếm ~150k / ~154k — gần như toàn bộ mẫu.
- Hai nhóm còn lại (`specific housing`, `terraced house`) sample <2k mỗi nhóm.
- Không có giá trị phân biệt thực tế.

### EMERGENCYSTATE_MODE
- `No` chiếm ~160k, `Yes` chỉ ~2k.
- Mặc dù `Yes` có default rate cao hơn (~9.5% vs ~7%), sample quá nhỏ để tin cậy.
- Tương tự HOUSETYPE_MODE — feature bị dominated bởi 1 giá trị.

---

## Tóm tắt quyết định

| Feature | Đánh giá | Hành động gợi ý |
|---|---|---|
| NAME_EDUCATION_TYPE | Mạnh | Giữ nguyên |
| NAME_INCOME_TYPE | Mạnh | Giữ, gộp nhóm sample nhỏ |
| NAME_HOUSING_TYPE | Mạnh | Giữ nguyên |
| CODE_GENDER | Mạnh | Giữ, xử lý XNA |
| NAME_CONTRACT_TYPE | Trung bình | Encode binary |
| NAME_FAMILY_STATUS | Trung bình | Giữ, xử lý Unknown |
| WALLSMATERIAL_MODE | Trung bình | Giữ, gộp nhóm hiếm |
| FLAG_OWN_CAR | Trung bình | Giữ tạm, kiểm tra lại sau |
| WEEKDAY_APPR_PROCESS_START | Yếu | Cân nhắc loại |
| FLAG_OWN_REALTY | Yếu | Cân nhắc loại |
| NAME_TYPE_SUITE | Yếu | Cân nhắc loại |
| FONDKAPREMONT_MODE | Yếu + missing nhiều | Loại |
| HOUSETYPE_MODE | Yếu | Loại |
| EMERGENCYSTATE_MODE | Yếu | Loại |

In [ ]:
# XEM XÉT CÁC CỘT BỊ EXCLUDED (có quá nhiều category, nên sẽ vẽ theo nhóm bin được tính theo trung bình default rate của category đó)


def plot_stats_binned(feature, n_bins=4, label_rotation=False, horizontal_layout=True):
    # Tính default rate từng category
    cat_perc = app_train.groupby(feature, as_index=False).agg(
        default_rate=("TARGET", "mean")
    )

    # Tạo bin động theo min/max của feature đó
    min_rate = cat_perc["default_rate"].min()
    max_rate = cat_perc["default_rate"].max()
    bins = np.linspace(min_rate, max_rate, n_bins + 1)

    # Tạo label dạng "8%-10%"
    labels = [f"{bins[i]*100:.0f}%-{bins[i+1]*100:.0f}%" for i in range(n_bins)]

    # Gán bin cho từng category
    cat_perc["bin"] = pd.cut(
        cat_perc["default_rate"], bins=bins, labels=labels, include_lowest=True
    )

    # Tính mean default rate theo bin để plot
    bin_perc = (
        cat_perc.groupby("bin", as_index=False)
        .agg(
            default_rate=("default_rate", "mean"),
            count=("default_rate", "count"),  # bao nhiêu category trong bin
        )
        .sort_values("default_rate")
    ).dropna()  # loại bỏ bin nào không có category nào

    # Số lượng hợp đồng gốc (theo category gốc, merge bin vào)
    temp = app_train[feature].value_counts().reset_index()
    temp.columns = [feature, "Số lượng"]
    temp = temp.merge(cat_perc[[feature, "bin"]], on=feature)
    bin_count = temp.groupby("bin", as_index=False)["Số lượng"].sum()

    if horizontal_layout:
        fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12, 6))
    else:
        fig, (ax1, ax2) = plt.subplots(nrows=2, figsize=(12, 14))

    sns.set_color_codes("pastel")

    # Biểu đồ số lượng
    sns.barplot(ax=ax1, x="bin", y="Số lượng", data=bin_count)
    ax1.set_title(f"Số lượng hợp đồng theo nhóm {feature}", fontsize=12)
    ax1.set_xlabel("Nhóm default rate")
    if label_rotation:
        ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha="right")

    # Biểu đồ default rate theo bin (thấp đến cao)
    sns.barplot(ax=ax2, x="bin", y="default_rate", order=bin_perc["bin"], data=bin_perc)
    ax2.set_title(f"Default rate theo nhóm {feature}", fontsize=12)
    ax2.set_ylabel("Default rate trung bình")
    ax2.set_xlabel("Nhóm default rate")
    if label_rotation:
        ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha="right")

    plt.tight_layout()
    plt.show()

    # In ra bảng để biết category nào thuộc bin nào
    print(cat_perc[[feature, "default_rate", "bin"]].sort_values("default_rate"))
    print("=" * 50)


# Chạy cho 2 cột bị excluded
excluded_columns = ["OCCUPATION_TYPE", "ORGANIZATION_TYPE"]
for col in excluded_columns:
    print(f"Biểu đồ phân bố cho cột: {col}")
    plot_stats_binned(col, n_bins=4, label_rotation=True)

# Insights từ phân nhóm theo Default Rate

## 1. OCCUPATION_TYPE (nghề nghiệp)

- **Phân bố rủi ro rất rõ ràng:** 4 nhóm default rate từ thấp đến cao (5–8%, 8–11%, 11–14%, 14–17%).
- **Nhóm an toàn nhất (5–8%)** gồm các nghề văn phòng, kỹ thuật, quản lý, y tế, kế toán – chiếm 10/18 nghề và lượng hợp đồng lớn (~84k).
- **Nhóm rủi ro cao nhất (14–17%)** chỉ có 1 nghề: `Low-skill Laborers` (lao động kỹ năng thấp) – default rate ~17,2%, số hợp đồng ~1k. Đây là nhóm cần kiểm soát rủi ro đặc biệt.
- **Xu hướng tăng dần** của default rate theo nhóm: 5–8% → 14–17% phản ánh đúng kỳ vọng về mức độ ổn định thu nhập.
- **Quy mô mẫu** ở nhóm rủi ro cao (~1k) khá nhỏ.

## 2. ORGANIZATION_TYPE (loại hình tổ chức)

- **Phân bố rủi ro cũng theo 4 nhóm:** 3–6%, 6–9%, 9–13% (thực tế default rate 10–12%), 13–16%.
- **Khác biệt về quy mô:** Nhóm rủi ro cao nhất (13–16%) chỉ có dưới ~2k hợp đồng, rất nhỏ so với các nhóm khác. Đây chỉ gồm 2 loại hình: `Industry: type 13` và `Transport: type 3` (default rate ~13,4% và ~15,8%).
- **Nhóm đông nhất** là 6–9% với ~158.000 hợp đồng – chiếm tỷ trọng chính, default rate ~7%.
- **Nhóm 3–6%** (rủi ro thấp) có ~78.000 hợp đồng, gồm các tổ chức ổn định như ngân hàng, trường học, quân đội, công an, bảo hiểm.
- **Rủi ro cao nhưng hiếm:** Vì nhóm 13–16% có quá ít mẫu, việc giữ nguyên 4 nhóm có thể gây overfitting hoặc mất ổn định khi chia train/test. Nên cân nhắc gộp nhóm này vào 9–13% (tạo bin `>9%`) hoặc tạo một flag nhị phân cho hai tổ chức đặc biệt.

## 3. So sánh hai biến

- `OCCUPATION_TYPE` có nhóm rủi ro cao **đúng thực tế** → phản ánh rủi ro tốt hơn nếu giữ nguyên các bin.
- `ORGANIZATION_TYPE` có nhiều category gốc hơn (57) nhưng phân bố mẫu lệch về mức rủi ro trung bình, nhóm rủi ro cao rất thưa thớt.

In [ ]:
print(app_train.dtypes.value_counts())
print("=" * 50)

# Cột số (int64, float64)
dtypes_numeric = app_train.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Tách thành 2 loại
# Lọc ra các cột có 2 giá trị duy nhất (binary) và chỉ chứa 0,1
binary_cols = [
    col
    for col in dtypes_numeric
    if app_train[col].nunique() == 2
    and set(app_train[col].dropna().unique()).issubset({0, 1})
]

# Lọc ra các cột còn lại là continuous
continuous_cols = [col for col in dtypes_numeric if col not in binary_cols]

print(f"Số cột binary: {len(binary_cols)}")
print(f"Số cột continuous: {len(continuous_cols)}")

In [ ]:
flag_document_cols = [col for col in binary_cols if "DOCUMENT" in col]

doc_stats = pd.DataFrame(
    {
        "So_luong_=1": app_train[flag_document_cols].sum(),
        "ty_le_=1": app_train[flag_document_cols].mean() * 100,
    }
).sort_values("ty_le_=1", ascending=False)

# print(doc_stats)

flag_doc_default = []

for col in flag_document_cols:
    rate_0 = app_train[app_train[col] == 0][
        "TARGET"
    ].mean()  # default rate khi không nộp giấy tờ
    rate_1 = app_train[app_train[col] == 1][
        "TARGET"
    ].mean()  # default rate khi nộp giấy tờ
    diff = abs(rate_1 - rate_0)
    flag_doc_default.append(
        {
            "cột": col,
            "default_rate_=0": round(rate_0, 4),
            "default_rate_=1": round(rate_1, 4),
            "chênh_lệch": round(diff, 4),
        }
    )

doc_default_df = pd.DataFrame(flag_doc_default).sort_values(
    "chênh_lệch", ascending=False
)
doc_default_df = doc_default_df.merge(doc_stats, left_on="cột", right_index=True)
print(doc_default_df)

# Insights từ phân tích FLAG_DOCUMENT

## Quan sát chính

### FLAG_DOCUMENT_2 — Cẩn thận với chênh lệch lớn
- Chênh lệch cao nhất (~23%) nhưng chỉ có **13 người** nộp.
- Sample quá nhỏ → không đáng tin cậy, nên **bỏ**.

### FLAG_DOCUMENT_4, FLAG_DOCUMENT_10, FLAG_DOCUMENT_12 — default_rate_=1 = 0%
- Không ai trong nhóm nộp giấy tờ này bị vỡ nợ — nghe có vẻ signal mạnh.
- Nhưng sample lần lượt chỉ có 25, 7, 2 người → kết quả ngẫu nhiên, không tin cậy, nên **bỏ**.

### FLAG_DOCUMENT_3 — Đảo chiều bất thường
- Hầu hết các cột: nộp giấy tờ (=1) → default rate thấp hơn.
- Riêng FLAG_DOCUMENT_3 thì ngược lại: nộp (=1) cao hơn không nộp (=0) (~8.8% vs ~6.2%).
- Sample lớn (71%) → kết quả đáng tin, đây là điểm bất thường **cần giữ lại** để model học.

### Phần lớn còn lại — chênh lệch quá nhỏ
- FLAG_DOCUMENT_5, FLAG_DOCUMENT_8, FLAG_DOCUMENT_20 chênh lệch gần 0 (<1%)
- Dù tỷ lệ nộp không quá thấp nhưng không có giá trị phân biệt → **bỏ**.

---

## Kết luận

| Cột | Tỷ lệ nộp | Chênh lệch default rate | Quyết định |
|---|---|---|---|
| FLAG_DOCUMENT_3 | 71% | 2.66% (đảo chiều) | Giữ |
| FLAG_DOCUMENT_6 | 8.8% | 2.75% | Giữ |
| FLAG_DOCUMENT_16 | 1% | 3.19% | Cân nhắc — sample nhỏ |
| Còn lại (16 cột) | <1% hoặc chênh lệch <2% | Không đáng kể | Bỏ |

In [ ]:
# Bỏ
id_col = ["SK_ID_CURR"]

# Nhóm số lần tra cứu tín dụng
amt_req_cols = [col for col in continuous_cols if "AMT_REQ" in col]

# Nhóm khoản vay
amt_cols = [col for col in continuous_cols if "AMT" in col and col not in amt_req_cols]

# Nhóm thời gian
day_cols = [col for col in continuous_cols if "DAYS" in col]

# Nhóm 4 External Source
ext_cols = [col for col in continuous_cols if "EXT_SOURCE" in col]

# Nhóm đặc điểm BĐS
avg_cols = [col for col in continuous_cols if col.endswith("_AVG")]
mode_cols = [col for col in continuous_cols if col.endswith("_MODE")]
medi_cols = [col for col in continuous_cols if col.endswith("_MEDI")]

# Nhóm còn lại
defined_cols = (
    id_col
    + amt_cols
    + amt_req_cols
    + day_cols
    + ext_cols
    + avg_cols
    + mode_cols
    + medi_cols
)
other_cols = [col for col in continuous_cols if col not in defined_cols]

# Kiểm tra cột
print("Cột ID: ", id_col)
print("Cột AMT: ", amt_cols)
print("Cột AMT_REQ: ", amt_req_cols)
print("Cột DAYS: ", day_cols)
print("Cột EXT_Source: ", ext_cols)
print("Cột AVG: ", avg_cols)
print("Cột MODE: ", mode_cols)
print("Cột MEDI: ", medi_cols)
print("Cột còn lại: ", other_cols)

In [ ]:
# Kiểm tra nhóm cột amt_cols
for col in amt_cols:
    print(f"\n{'='*50}")
    print(f"Cột: {col}")
    print(f"{'='*50}")

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # phân phối của cột
    sns.histplot(app_train[col], ax=axes[0], kde=True)
    axes[0].set_title(f"Phân phối của {col}")
    axes[0].set_xlabel(col)

    # boxplot
    sns.boxplot(x=app_train[col], ax=axes[1])
    axes[1].set_title(f"Boxplot của {col}")

    # phân phối theo target
    sns.kdeplot(data=app_train, x=col, hue="TARGET", common_norm=False, ax=axes[2])
    axes[2].set_title(f"Phân phối của {col} theo TARGET")

    plt.tight_layout()
    plt.show()

    # Thống kê cơ bản
    print(app_train[col].describe().round(2))

    # Correlation với target
    corr = app_train[col].corr(app_train["TARGET"])
    print(f"\n Tương quan với TARGET: {corr: .4f}")

In [ ]:
# Function để chia các giá trị liên tục thành bin rồi tính default rate theo bin đó


def plot_stats_binned_numeric(
    feature, n_bins=10, label_rotation=False, horizontal_layout=True
):
    # Cắt outlier theo quantile 1% - 99%
    q_low = app_train[feature].quantile(0.01)
    q_high = app_train[feature].quantile(0.99)
    # Nén data về khoảng này để tránh outlier ảnh hưởng quá nhiều đến việc chia bin
    data_clipped = app_train[feature].clip(q_low, q_high)

    # Chia bin theo giá trị feature
    app_train[f"{feature}_bin"] = pd.cut(data_clipped, bins=n_bins)

    # Tính số lượng và defaul rate theo bin
    bin_stats = (
        app_train.groupby(f"{feature}_bin", as_index=False)
        .agg(so_luong=(feature, "count"), default_rate=("TARGET", "mean"))
        .dropna()
    )

    if horizontal_layout:
        fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12, 6))
    else:
        fig, (ax1, ax2) = plt.subplots(nrows=2, figsize=(12, 14))

    sns.set_color_codes("pastel")

    # Biểu đồ số lượng
    sns.barplot(ax=ax1, x=f"{feature}_bin", y="so_luong", data=bin_stats)
    ax1.set_title(f"Số lượng theo {feature}", fontsize=12)
    ax1.set_xlabel(f"Khoảng giá trị")
    ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha="right")

    # Biểu đồ default rate
    sns.barplot(ax=ax2, x=f"{feature}_bin", y="default_rate", data=bin_stats)
    ax2.set_title(f"Default rate theo {feature}", fontsize=12)
    ax2.set_ylabel("Default rate")
    ax2.set_xlabel(f"Khoảng giá trị")
    ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha="right")

    plt.tight_layout()
    plt.show()

    # Xóa cột tạm
    app_train.drop(columns=[f"{feature}_bin"], inplace=True)

    print(bin_stats)
    print("=" * 50)

In [ ]:
for col in amt_cols:
    print(f"Biểu đồ phân bố cho cột: {col}")
    plot_stats_binned_numeric(col, n_bins=10, label_rotation=True)

# Insights từ phân tích nhóm AMT (Tài chính khoản vay)

## 1. AMT_CREDIT (Số tiền vay)
- **Xu hướng rõ ràng:** Vay càng nhiều → default rate càng thấp (từ ~8.2% xuống ~2.5% ở vùng trung bình).
- **Vùng rủi ro cao nhất:** Khoản vay nhỏ (~40k-450k) chiếm đa số mẫu (~238k/307k) và có default rate cao nhất (~8-9%).
- **Vùng outlier (>2.8 triệu):** Chỉ ~43 người, default rate không đáng tin cậy.
- **Kết luận:** Feature có signal tốt, quan hệ gần tuyến tính từ vùng trung bình trở lên. Giữ lại.

## 2. AMT_INCOME_TOTAL (Thu nhập)
- **Xu hướng chính:** Thu nhập càng cao → default rate càng thấp (~8.6% xuống ~5.6%).
- **Điểm bất thường:** Bin thu nhập thấp nhất (~8.16%) lại thấp hơn bin kế tiếp (~8.58%) — có thể do nhóm này vay ít, khoản nhỏ dễ trả hơn, hoặc do ảnh hưởng của clip ở vùng biên.
- **Quan hệ:** Gần tuyến tính từ bin 1 trở đi.
- **Kết luận:** Feature có signal, giữ lại.

## 3. AMT_ANNUITY (Khoản trả góp hàng tháng)
- **Hình dạng phi tuyến rõ ràng:** Tăng từ bin 0 đến bin 3 (~9.8%) rồi giảm dần về bin 9 (~3.3%).
- **Vùng rủi ro cao nhất:** Trả góp ~25k-32k/tháng — ngưỡng "vừa đủ trả", không quá thấp cũng không quá cao.
- **Hai đầu đều thấp:** Trả góp rất nhỏ (vay ít, dễ trả) và rất lớn (thu nhập cao, thẩm định kỹ) đều ít rủi ro hơn.
- **Kết luận:** Signal tốt nhưng phi tuyến → correlation thấp không phản ánh đúng. Tree-based model sẽ bắt được pattern này. Giữ lại.

## 4. AMT_GOODS_PRICE (Giá trị hàng hóa)
- **Bin 0-2:** Default rate tăng dần, đạt đỉnh ở ~414k-587k (~9.8%) — nhóm mua hàng trung bình có thể đang chi tiêu vượt khả năng.
- **Bin 3-9:** Giảm đều và nhất quán từ ~7.8% xuống ~3.8% — người mua hàng đắt tiền có tài chính ổn định hơn.
- **Pattern tương tự AMT_CREDIT:** Hợp lý vì 2 cột này thường tương quan cao với nhau.
- **Kết luận:** Signal tốt, quan hệ phi tuyến. Giữ lại.

---

## Tóm tắt nhóm AMT

| Feature | Quan hệ với TARGET | Signal | Quyết định |
|---|---|---|---|
| AMT_CREDIT | Gần tuyến tính, vay nhiều rủi ro thấp | Tốt | Giữ |
| AMT_INCOME_TOTAL | Gần tuyến tính, thu nhập cao rủi ro thấp | Trung bình | Giữ |
| AMT_ANNUITY | Phi tuyến, đỉnh ở mức trung bình | Tốt | Giữ |
| AMT_GOODS_PRICE | Phi tuyến, đỉnh ở mức trung bình | Tốt | Giữ |

**Lưu ý chung:** Correlation tuyến tính thấp không phản ánh đúng signal thực sự của các cột AMT — cần dùng bin analysis hoặc tree-based model để khai thác đúng.

In [ ]:
# Xử lý cột DAYS_

for col in day_cols:
    app_train[col] = app_train[col].abs()

app_train[day_cols].describe().round(2)

In [ ]:
# Kiểm tra nhóm cột day_cols
for col in day_cols:
    print(f"\n{'='*50}")
    print(f"Cột: {col}")
    print(f"{'='*50}")

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # phân phối của cột
    sns.histplot(app_train[col], ax=axes[0], kde=True)
    axes[0].set_title(f"Phân phối của {col}")
    axes[0].set_xlabel(col)

    # boxplot
    sns.boxplot(x=app_train[col], ax=axes[1])
    axes[1].set_title(f"Boxplot của {col}")

    # phân phối theo target
    sns.kdeplot(data=app_train, x=col, hue="TARGET", common_norm=False, ax=axes[2])
    axes[2].set_title(f"Phân phối của {col} theo TARGET")

    plt.tight_layout()
    plt.show()

    # Thống kê cơ bản
    print(app_train[col].describe().round(2))

    # Correlation với target
    corr = app_train[col].corr(app_train["TARGET"])
    print(f"\n Tương quan với TARGET: {corr: .4f}")

In [ ]:
# 365243 ~ 1000 năm, thất nghiệp hoặc chưa đi làm hoặc không xác định
print((app_train["DAYS_EMPLOYED"] == 365243).sum())

# Tạo cột mới, quét toàn bộ day_employed, nếu có giá trị nào bằng 365243 thì gán 1, ngược lại gán 0
app_train["DAYS_EMPLOYED_ANOM"] = (app_train["DAYS_EMPLOYED"] == 365243).astype(int)

# Thay 365243 bằng giá trị NaN để không ảnh hưởng đến việc chia bin
app_train["DAYS_EMPLOYED"] = app_train["DAYS_EMPLOYED"].replace(365243, np.nan)

# Chuyển ngày thành năm
for col in day_cols:
    app_train[col] = app_train[col] / 365

# Xem tỉ lệ vỡ nợ của nhóm có việc và không việc
print("Tỷ lệ vỡ nợ của nhóm không việc:")
print(app_train.groupby("DAYS_EMPLOYED_ANOM")["TARGET"].mean().round(4))

In [ ]:
for col in day_cols:
    print(f"Biểu đồ phân bố cho cột: {col}")
    plot_stats_binned_numeric(col, n_bins=10, label_rotation=True)

# Insights từ phân tích nhóm DAYS (Thời gian)

## Đặc điểm chung
Tất cả các cột DAYS đều có **mối quan hệ tuyến tính khá rõ** với TARGET — thời gian càng dài thì default rate càng thấp. Đây là nhóm feature có signal nhất quán nhất so với các nhóm đã phân tích.

**Lưu ý chung:** Nhóm có thời gian dài (nhiều năm kinh nghiệm, ổn định lâu dài) thường có **ít mẫu hơn** so với nhóm mới — cần lưu ý khi model học từ nhóm này.

---

## 1. DAYS_BIRTH (Tuổi)
- Xu hướng tuyến tính rõ ràng nhất: tuổi càng cao → default rate càng thấp (11.5% → 4.7%).
- Người trẻ (22-27 tuổi) rủi ro cao nhất — thu nhập chưa ổn định, ít kinh nghiệm tài chính.
- Người lớn tuổi (62-67 tuổi) rủi ro thấp nhất — nhiều năm kinh nghiệm, thu nhập ổn định hơn.
- **Kết luận:** Feature mạnh nhất trong nhóm DAYS. Giữ lại.

---

## 2. DAYS_EMPLOYED (Số năm đi làm)
- Xu hướng rõ ràng: đi làm càng lâu → default rate càng thấp (11% → 3.4%).
- **Điểm đáng chú ý:** Số lượng mẫu giảm mạnh sau mốc 15 năm (~12,983 người) so với nhóm mới đi làm (~99,842 người) — người làm lâu năm ít vay hơn hẳn.
- **Giá trị bất thường 365243:** Đã xử lý thành flag riêng `DAYS_EMPLOYED_ANOM` — nhóm này (hưu trí/nội trợ) có default rate thấp hơn (~5.4% vs ~8.66%), bản thân flag là feature có giá trị.
- **Kết luận:** Giữ cả `DAYS_EMPLOYED` và `DAYS_EMPLOYED_ANOM`.

---

## 3. DAYS_REGISTRATION (Số năm kể từ khi đăng ký giấy tờ)
- Xu hướng giảm dần nhưng **không đều** ở các bin đầu (bin 1-3 gần bằng nhau ~8.6-8.7%) trước khi giảm rõ từ bin 4 trở đi.
- Người ít đổi chỗ ở (đăng ký lâu năm) có default rate thấp hơn (~5.1% vs ~9.3%).
- Signal yếu hơn so với DAYS_BIRTH và DAYS_EMPLOYED ở vùng trung bình.
- **Kết luận:** Giữ lại, signal có nhưng không mạnh bằng các cột khác.

---

## 4. DAYS_ID_PUBLISH (Số năm kể từ khi cấp CMND)
- Xu hướng giảm dần khá nhất quán: ~10.2% → ~6.2%.
- **Điểm bất thường nhỏ:** Bin 9 (13.4-14.9 năm) tăng nhẹ so với bin 8 (~6.2% vs ~6.04%) — có thể do sample nhỏ hơn ở vùng này.
- Thông tin cá nhân càng lâu → ít thay đổi nhân thân → ổn định hơn → ít rủi ro hơn.
- **Kết luận:** Giữ lại.

---

## 5. DAYS_LAST_PHONE_CHANGE (Số năm kể từ lần cuối đổi SĐT)
- Xu hướng tuyến tính rõ và nhất quán nhất trong nhóm: ~9.6% → ~4.8%.
- Dùng SĐT càng lâu → ổn định hơn → ít rủi ro hơn.
- Phân bố mẫu lệch về nhóm mới đổi SĐT (~83k người trong bin 0).
- **Kết luận:** Giữ lại.

---

## Tóm tắt nhóm DAYS

| Feature | Quan hệ với TARGET | Signal | Quyết định |
|---|---|---|---|
| DAYS_BIRTH | Tuyến tính, mạnh nhất nhóm | Mạnh | Giữ |
| DAYS_EMPLOYED | Tuyến tính, có giá trị bất thường | Mạnh | Giữ + giữ flag ANOM |
| DAYS_REGISTRATION | Gần tuyến tính, yếu hơn | Trung bình | Giữ |
| DAYS_ID_PUBLISH | Gần tuyến tính | Trung bình | Giữ |
| DAYS_LAST_PHONE_CHANGE | Tuyến tính, nhất quán | Tốt | Giữ |

**Lưu ý chung:** Các cột DAYS đã được convert sang đơn vị năm (chia 365) và lấy giá trị tuyệt đối để dễ đọc hơn.


In [ ]:
# Nhóm cột ext_cols, việc missing có thể tới từ việc thiếu dữ liệu của khách hàng
print(app_train[ext_cols].describe().round(4))
print("=" * 50)
print(app_train[ext_cols].isnull().sum())
print("=" * 50)

for col in ext_cols:
    app_train[f"{col}_MISSING"] = app_train[col].isna().astype(int)
    print("Tỷ lệ vỡ nợ của nhóm không có thông tin tín dụng từ bên ngoài:")
    print(app_train.groupby(f"{col}_MISSING")["TARGET"].mean().round(4))
    print("=" * 50)

In [ ]:
# Cột ext2_missing không có ý nghĩa nên bỏ
app_train.drop(columns=["EXT_SOURCE_2_MISSING"], inplace=True)

In [ ]:
for col in ext_cols:
    print(f"Biểu đồ phân bố cho cột: {col}")
    plot_stats_binned_numeric(col, n_bins=10, label_rotation=True)

# Insights từ phân tích nhóm EXT_SOURCE (Điểm tín dụng bên ngoài)

## Đặc điểm chung
- Giá trị nằm trong khoảng 0-1, mean ~0.5 cho cả 3 cột → đã được chuẩn hóa sẵn.
- **Điểm càng cao → default rate càng thấp** — xu hướng tuyến tính mạnh và nhất quán nhất so với tất cả các nhóm đã phân tích.

---

## 1. EXT_SOURCE_1
- Default rate giảm mạnh từ ~20% (điểm thấp) xuống ~2.4% (điểm cao) — chênh lệch ~8x.
- **Missing: 173,378 dòng (~56%)** — đây là vấn đề nghiêm trọng nhất trong toàn bộ dataset.
- Nhóm missing có default rate cao hơn nhóm có điểm (8.52% vs 7.5%) → missing không phải ngẫu nhiên, có thể do khách hàng chưa có lịch sử tín dụng với tổ chức này.
- **Kết luận:** Giữ cả `EXT_SOURCE_1` và flag `EXT_SOURCE_1_MISSING`.

## 2. EXT_SOURCE_2
- Default rate giảm từ ~23% xuống ~3.2% — signal mạnh và đều nhất trong 3 cột.
- **Missing chỉ 660 dòng (~0.2%)** → gần như không đáng kể.
- Nhóm missing có default rate gần bằng nhóm có điểm (7.88% vs 8.07%) → missing ở đây có thể là ngẫu nhiên, không mang thông tin.
- **Kết luận:** Giữ `EXT_SOURCE_2`, bỏ flag `EXT_SOURCE_2_MISSING`.

## 3. EXT_SOURCE_3
- Default rate giảm từ ~24% xuống ~3.2% — signal mạnh tương đương EXT_SOURCE_2.
- **Missing: 60,965 dòng (~20%)**.
- Nhóm missing có default rate cao hơn rõ ràng (9.31% vs 7.77%) → missing có ý nghĩa thực sự.
- **Kết luận:** Giữ cả `EXT_SOURCE_3` và flag `EXT_SOURCE_3_MISSING`.

---

## Xử lý Missing

| Cột | Missing | Ý nghĩa missing | Hành động |
|---|---|---|---|
| EXT_SOURCE_1 | ~56% | Có signal (8.52% vs 7.5%) | Giữ + tạo flag |
| EXT_SOURCE_2 | ~0.2% | Không có signal | Impute bằng median |
| EXT_SOURCE_3 | ~20% | Có signal (9.31% vs 7.77%) | Giữ + tạo flag |

---

## Tóm tắt nhóm EXT_SOURCE

| Feature | Signal | Missing | Quyết định |
|---|---|---|---|
| EXT_SOURCE_1 | Rất mạnh | ~56% | Giữ + flag MISSING |
| EXT_SOURCE_2 | Rất mạnh | ~0.2% | Giữ, impute median |
| EXT_SOURCE_3 | Rất mạnh | ~20% | Giữ + flag MISSING |

**Lưu ý:** Đây là nhóm feature quan trọng nhất trong toàn bộ dataset — cả 3 cột đều có signal mạnh và nhất quán. Cần xử lý missing cẩn thận trước khi đưa vào model.

In [ ]:
# Nhóm cột mode_cols, việc missing có thể tới từ việc thiếu dữ liệu của khách hàng
print(app_train[mode_cols].describe().round(4))
print("=" * 50)
print(app_train[mode_cols].isnull().sum())
print("=" * 50)

for col in mode_cols:
    app_train[f"{col}_MISSING"] = app_train[col].isna().astype(int)
    print("Tỷ lệ vỡ nợ của nhóm không có thông tin BĐS:")
    print(app_train.groupby(f"{col}_MISSING")["TARGET"].mean().round(4))
    print("=" * 50)

In [ ]:
# Kết quả cho thấy, việc ko có thông tin về BĐS sẽ có tỷ lệ vỡ nợ cao hơn,
# nhưng gắn cờ 15 cột là k cần thiết
app_train["BDS_INFO_MISSING"] = app_train[mode_cols].isnull().any(axis=1).astype(int)
print("Tỷ lệ vỡ nợ của nhóm không có thông tin BĐS:")
print(app_train.groupby("BDS_INFO_MISSING")["TARGET"].mean().round(4))

# Bỏ các cột missing tạo ở bước trước
missing_cols = [f"{col}_MISSING" for col in mode_cols]
app_train.drop(columns=missing_cols, inplace=True)

In [ ]:
for col in mode_cols:
    print(f"Biểu đồ phân bố cho cột: {col}")
    plot_stats_binned_numeric(col, n_bins=10, label_rotation=True)

In [ ]:
# Nhóm cột req_col, việc missing có thể tới từ việc thiếu dữ liệu của khách hàng
print(app_train[amt_req_cols].describe().round(4))
print("=" * 50)
print(app_train[amt_req_cols].isnull().sum())
print("=" * 50)

for col in amt_req_cols:
    app_train[f"{col}_MISSING"] = app_train[col].isna().astype(int)
    print("Tỷ lệ vỡ nợ của nhóm không có thông tin:")
    print(app_train.groupby(f"{col}_MISSING")["TARGET"].mean().round(4))
    print("=" * 50)

In [ ]:
# Kết quả cho thấy, việc ko có thông tin về tín dụng sẽ có tỷ lệ vỡ nợ cao hơn,
# nhưng gắn cờ tất cả cột là k cần thiết
app_train["AMT_REQ_MISSING"] = app_train[amt_req_cols].isnull().any(axis=1).astype(int)
print("Tỷ lệ vỡ nợ của nhóm không có thông tin tín dụng:")
print(app_train.groupby("AMT_REQ_MISSING")["TARGET"].mean().round(4))

# Bỏ các cột missing tạo ở bước trước
missing_req_cols = [f"{col}_MISSING" for col in amt_req_cols]
app_train.drop(columns=missing_req_cols, inplace=True)

In [ ]:
# Nhóm cột other_col, việc missing có thể tới từ việc thiếu dữ liệu của khách hàng
print(app_train[other_cols].describe().round(4))
print("=" * 50)
print(app_train[other_cols].isnull().sum())
print("=" * 50)

for col in other_cols:
    app_train[f"{col}_MISSING"] = app_train[col].isna().astype(int)
    print("Tỷ lệ vỡ nợ của nhóm không có thông tin:")
    print(app_train.groupby(f"{col}_MISSING")["TARGET"].mean().round(4))
    print("=" * 50)

In [ ]:
# Cột không missing
no_missing_cols = [
    "CNT_CHILDREN",
    "REGION_POPULATION_RELATIVE",
    "REGION_RATING_CLIENT",
    "REGION_RATING_CLIENT_W_CITY",
    "HOUR_APPR_PROCESS_START",
]
drop_flags_1 = [f"{col}_MISSING" for col in no_missing_cols]

# Cột missing 2 giá trị, điền median
app_train["CNT_FAM_MEMBERS"] = app_train["CNT_FAM_MEMBERS"].fillna(
    app_train["CNT_FAM_MEMBERS"].median()
)
drop_flags_2 = ["CNT_FAM_MEMBERS_MISSING"]

# Cột own_car, fillna = median, cột flag vẫn giữ
app_train["OWN_CAR_AGE"] = app_train["OWN_CAR_AGE"].fillna(
    app_train["OWN_CAR_AGE"].median()
)

# Social Circle, vòng quan hệ của khách
social_cols = [
    "OBS_30_CNT_SOCIAL_CIRCLE",
    "OBS_60_CNT_SOCIAL_CIRCLE",
    "DEF_30_CNT_SOCIAL_CIRCLE",
    "DEF_60_CNT_SOCIAL_CIRCLE",
]
drop_flags_3 = [f"{col}_MISSING" for col in social_cols]

# Bỏ các cột flag k cần
app_train.drop(columns=drop_flags_1 + drop_flags_2 + drop_flags_3, inplace=True)

print("Các flag còn lại: ")
flag_cols = [col for col in app_train.columns if "MISSING" in col or "ANOM" in col]
print(flag_cols)

In [ ]:
for col in other_cols + amt_req_cols:
    print(f"Biểu đồ phân bố cho cột: {col}")
    plot_stats_binned_numeric(col, n_bins=10)

In [ ]:
# Chia số nguyên thành thập phân không phù hợp nên chia lại theo số nguyên

int_cols = [
    "CNT_CHILDREN",
    "CNT_FAM_MEMBERS",
    "REGION_RATING_CLIENT",
    "REGION_RATING_CLIENT_W_CITY",
    "HOUR_APPR_PROCESS_START",
    "OBS_30_CNT_SOCIAL_CIRCLE",
    "DEF_30_CNT_SOCIAL_CIRCLE",
    "OBS_60_CNT_SOCIAL_CIRCLE",
    "DEF_60_CNT_SOCIAL_CIRCLE",
]

for col in int_cols:
    print(f"\n{col}:")
    print(f"Số giá trị duy nhất: {app_train[col].nunique()}")

for col in int_cols:
    stats = (
        app_train.groupby(col)
        .agg(so_luong=(col, "count"), default_rate=("TARGET", "mean"))
        .round(4)
    )
    print(f"\n{col}:")
    print(stats)
    print("=" * 50)

# Insights từ nhóm Other Cols và AMT_REQ

## 1. REGION_POPULATION_RELATIVE (Mật độ dân số khu vực)
- Không có xu hướng tuyến tính rõ ràng — default rate dao động không nhất quán.
- Bin cuối (mật độ cao nhất) có default rate thấp nhất (~4%) nhưng pattern trung gian không rõ.
- **Kết luận:** Signal yếu, cân nhắc loại bỏ.

---

## 2. OWN_CAR_AGE (Tuổi xe)
- Xe mới (<7 năm) default rate thấp nhất (~5.7%) — người có xe mới thường tài chính tốt hơn.
- Xe càng cũ (7-45 năm) default rate tăng dần từ ~8.4% đến ~11.1%.
- Vùng xe rất cũ (>45 năm) sample cực nhỏ (<20 người) → không tin cậy.
- **Kết luận:** Signal có nhưng cần lưu ý vùng outlier. Giữ lại cùng flag `OWN_CAR_AGE_MISSING`.

---

## 3. CNT_CHILDREN (Số con)
- Chỉ tin vào vùng 0-4 con (sample đủ lớn):
  - 0 con: ~7.7% — thấp nhất
  - 1-2 con: ~8.7-8.9%
  - 3-4 con: ~9.6-12.8% — cao nhất
- Từ 5 con trở lên sample quá nhỏ (<100 người), default rate dao động 0-100% → không tin cậy.
- **Kết luận:** Signal trung bình ở vùng 0-4. Giữ lại.

---

## 4. CNT_FAM_MEMBERS (Số thành viên gia đình)
- Pattern tương tự CNT_CHILDREN: gia đình càng đông → default rate tăng.
- Chỉ tin vào vùng 1-6 người (sample đủ lớn).
- Tương quan cao với CNT_CHILDREN → thông tin trùng lặp.
- **Kết luận:** Cân nhắc bỏ, giữ CNT_CHILDREN là đủ.

---

## 5. REGION_RATING_CLIENT và REGION_RATING_CLIENT_W_CITY (Xếp hạng khu vực)
- Chỉ có 3 giá trị (1, 2, 3) — thực chất là ordinal feature.
- Pattern rõ ràng và nhất quán:
  - Rating 1: ~4.8% (khu vực tốt nhất)
  - Rating 2: ~7.9%
  - Rating 3: ~11-11.4% (khu vực rủi ro nhất)
- 2 cột gần như giống hệt nhau về default rate.
- **Kết luận:** Signal tốt, chỉ cần giữ 1 — gợi ý giữ `REGION_RATING_CLIENT_W_CITY` vì chi tiết hơn.

---

## 6. HOUR_APPR_PROCESS_START (Giờ nộp đơn)
- Giờ 0, 22, 23 có default rate cao (~10-15%) nhưng sample cực nhỏ (<200 người) → không tin cậy.
- Trong giờ làm việc thực tế (9-17h): default rate giảm dần từ ~8.2% xuống ~7.5% — pattern nhẹ.
- Sáng sớm (5-7h) rủi ro cao hơn (~10-11%) — có thể nhóm bị áp lực tài chính cấp bách.
- **Kết luận:** Signal yếu, cân nhắc loại bỏ.

---

## 7. OBS_30 và OBS_60 CNT_SOCIAL_CIRCLE (Số người quan sát trong vòng tròn xã hội)
- Chỉ tin vào vùng 0-14 (sample đủ lớn).
- Default rate tăng nhẹ từ ~7.9% đến ~10.7% nhưng không nhất quán — dao động lên xuống.
- OBS_30 và OBS_60 gần như giống hệt nhau.
- **Kết luận:** Signal yếu, bỏ cả 2 cột.

---

## 8. DEF_30 và DEF_60 CNT_SOCIAL_CIRCLE (Số người vỡ nợ trong vòng tròn xã hội)
- Pattern tăng dần rõ và nhất quán ở vùng 0-5 (sample đủ lớn):
  - DEF_30: 0 người (~7.8%) → 5 người (~14.3%)
  - DEF_60: 0 người (~7.8%) → 5 người (~15%)
- Từ giá trị 6 trở lên sample quá nhỏ → không tin cậy.
- DEF_30 và DEF_60 có pattern gần giống nhau.
- **Kết luận:** Signal tốt, chỉ giữ 1 trong 2 — gợi ý giữ `DEF_60_CNT_SOCIAL_CIRCLE` vì window dài hơn thường ổn định hơn.

---

## 9. AMT_REQ_CREDIT_BUREAU (Số lần tra cứu tín dụng)
- `HOUR` và `DAY`: Gần như toàn bộ = 0, không có signal → **bỏ**.
- `WEEK`: Tương tự, gần như toàn 0 → **bỏ**.
- `MON`: Tra cứu nhiều hơn lại có default rate thấp hơn — ngược kỳ vọng, signal không nhất quán → **bỏ**.
- `QRT`: Không nhất quán → **bỏ**.
- `YEAR`: Xu hướng rõ nhất — tra cứu càng nhiều/năm → default rate tăng dần (~7.1% → ~9.4%). Phản ánh người đang tích cực tìm vay nhiều nơi → rủi ro cao hơn. **Giữ lại**.
- **Kết luận:** Chỉ giữ `AMT_REQ_CREDIT_BUREAU_YEAR`.

---

## Tóm tắt quyết định

| Feature | Signal | Quyết định |
|---|---|---|
| REGION_POPULATION_RELATIVE | Yếu | Cân nhắc bỏ |
| OWN_CAR_AGE | Trung bình | Giữ + flag MISSING |
| CNT_CHILDREN | Trung bình | Giữ |
| CNT_FAM_MEMBERS | Yếu (trùng CNT_CHILDREN) | Bỏ |
| REGION_RATING_CLIENT | Tốt | Bỏ (trùng W_CITY) |
| REGION_RATING_CLIENT_W_CITY | Tốt | Giữ |
| HOUR_APPR_PROCESS_START | Yếu | Bỏ |
| OBS_30_CNT_SOCIAL_CIRCLE | Yếu | Bỏ |
| OBS_60_CNT_SOCIAL_CIRCLE | Yếu | Bỏ |
| DEF_30_CNT_SOCIAL_CIRCLE | Trung bình | Bỏ (trùng DEF_60) |
| DEF_60_CNT_SOCIAL_CIRCLE | Trung bình | Giữ |
| AMT_REQ_CREDIT_BUREAU_HOUR | Không có | Bỏ |
| AMT_REQ_CREDIT_BUREAU_DAY | Không có | Bỏ |
| AMT_REQ_CREDIT_BUREAU_WEEK | Không có | Bỏ |
| AMT_REQ_CREDIT_BUREAU_MON | Yếu | Bỏ |
| AMT_REQ_CREDIT_BUREAU_QRT | Yếu | Bỏ |
| AMT_REQ_CREDIT_BUREAU_YEAR | Trung bình | Giữ |

# Tóm tắt EDA và Quyết định Feature Selection — app_train

## 1. Tổng quan dataset
- **307,511 dòng**, **122 cột** ban đầu (sau khi bỏ SK_ID_CURR còn 121 cột)
- **TARGET**: 8.07% vỡ nợ — mất cân bằng class rõ ràng
- Dtype: 65 float64, 41 int64, 16 object

---

## 2. Nhóm Object (Categorical)

### Đã xử lý
- Loại 2 cột có quá nhiều giá trị (`OCCUPATION_TYPE`, `ORGANIZATION_TYPE`) khỏi plot thông thường → phân tích riêng bằng bin theo default rate
- Áp dụng bin động (min-max theo từng feature) để hiển thị EDA

### Quyết định

| Feature | Quyết định |
|---|---|
| NAME_EDUCATION_TYPE | Giữ |
| NAME_INCOME_TYPE | Giữ, gộp nhóm sample nhỏ (Maternity leave + Unemployed) |
| NAME_HOUSING_TYPE | Giữ |
| CODE_GENDER | Giữ, xử lý XNA |
| NAME_CONTRACT_TYPE | Giữ, encode binary |
| NAME_FAMILY_STATUS | Giữ, xử lý Unknown |
| WALLSMATERIAL_MODE | Giữ, gộp nhóm hiếm |
| FLAG_OWN_CAR | Giữ |
| OCCUPATION_TYPE | Giữ |
| ORGANIZATION_TYPE | Giữ |
| WEEKDAY_APPR_PROCESS_START | Bỏ |
| FLAG_OWN_REALTY | Bỏ |
| NAME_TYPE_SUITE | Bỏ |
| FONDKAPREMONT_MODE | Bỏ |
| HOUSETYPE_MODE | Bỏ |
| EMERGENCYSTATE_MODE | Bỏ |

---

## 3. Nhóm Binary (FLAG)

### FLAG_DOCUMENT
| Feature | Quyết định |
|---|---|
| FLAG_DOCUMENT_3 | Giữ (sample lớn, có signal đảo chiều) |
| FLAG_DOCUMENT_6 | Giữ |
| FLAG_DOCUMENT_16 | Cân nhắc (sample nhỏ) |
| FLAG_DOCUMENT_21 | Cân nhắc (chênh lệch 5.5%) |
| Còn lại (16 cột) | Bỏ |

### FLAG liên lạc
| Feature | Quyết định |
|---|---|
| FLAG_MOBIL | Giữ |
| FLAG_EMP_PHONE | Giữ |
| FLAG_WORK_PHONE | Giữ |
| FLAG_CONT_MOBILE | Giữ |
| FLAG_PHONE | Giữ |
| FLAG_EMAIL | Giữ |

### FLAG địa chỉ
| Feature | Quyết định |
|---|---|
| REG_REGION_NOT_LIVE_REGION | Giữ |
| REG_REGION_NOT_WORK_REGION | Giữ |
| LIVE_REGION_NOT_WORK_REGION | Giữ |
| REG_CITY_NOT_LIVE_CITY | Giữ |
| REG_CITY_NOT_WORK_CITY | Giữ |
| LIVE_CITY_NOT_WORK_CITY | Giữ |

---

## 4. Nhóm AMT (Tài chính khoản vay)

| Feature | Signal | Quyết định |
|---|---|---|
| AMT_CREDIT | Tốt, gần tuyến tính | Giữ |
| AMT_INCOME_TOTAL | Trung bình, gần tuyến tính | Giữ |
| AMT_ANNUITY | Tốt, phi tuyến | Giữ |
| AMT_GOODS_PRICE | Tốt, phi tuyến | Giữ |

**Lưu ý:** Correlation thấp không phản ánh đúng signal thực — cần bin analysis hoặc tree-based model.

---

## 5. Nhóm DAYS (Thời gian)

| Feature | Signal | Quyết định |
|---|---|---|
| DAYS_BIRTH | Mạnh, tuyến tính | Giữ |
| DAYS_EMPLOYED | Mạnh, tuyến tính | Giữ + flag ANOM |
| DAYS_REGISTRATION | Trung bình | Giữ |
| DAYS_ID_PUBLISH | Trung bình | Giữ |
| DAYS_LAST_PHONE_CHANGE | Tốt, tuyến tính | Giữ |

**Xử lý đặc biệt:**
- Tất cả đã convert sang số dương và chia 365 (đơn vị năm)
- `DAYS_EMPLOYED = 365243` → tạo flag `DAYS_EMPLOYED_ANOM` (hưu trí/nội trợ, default rate 5.4% vs 8.66%)

---

## 6. Nhóm EXT_SOURCE (Điểm tín dụng bên ngoài)

| Feature | Signal | Missing | Quyết định |
|---|---|---|---|
| EXT_SOURCE_1 | Rất mạnh | ~56% | Giữ + flag MISSING |
| EXT_SOURCE_2 | Rất mạnh | ~0.2% | Giữ, impute median |
| EXT_SOURCE_3 | Rất mạnh | ~20% | Giữ + flag MISSING |

**Lưu ý:** Đây là nhóm feature quan trọng nhất trong dataset.

---

## 7. Nhóm BĐS (_MODE)

| Quyết định | Chi tiết |
|---|---|
| Chọn nhóm _MODE | Vì TOTALAREA chỉ có _MODE, dùng _MODE cho đồng nhất |
| Bỏ _AVG và _MEDI | Thông tin trùng lặp |
| Tạo flag chung | `BDS_INFO_MISSING` thay cho 15 flag riêng lẻ (6.96% vs 8.48%) |
| Model giải thích | Chọn vài cột đại diện (TOTALAREA_MODE, YEARS_BUILD_MODE, FLOORSMAX_MODE) |
| Model dự đoán | Giữ nguyên 15 cột |

---

## 8. Nhóm Other và AMT_REQ

| Feature | Quyết định |
|---|---|
| OWN_CAR_AGE | Giữ + flag MISSING |
| CNT_CHILDREN | Giữ |
| REGION_RATING_CLIENT_W_CITY | Giữ |
| DEF_60_CNT_SOCIAL_CIRCLE | Giữ |
| AMT_REQ_CREDIT_BUREAU_YEAR | Giữ |
| AMT_REQ_MISSING | Giữ flag (10.34% vs 7.72%) |
| REGION_POPULATION_RELATIVE | Cân nhắc bỏ |
| HOUR_APPR_PROCESS_START | Bỏ |
| CNT_FAM_MEMBERS | Bỏ |
| REGION_RATING_CLIENT | Bỏ |
| OBS_30/60_CNT_SOCIAL_CIRCLE | Bỏ |
| DEF_30_CNT_SOCIAL_CIRCLE | Bỏ |
| AMT_REQ_CREDIT_BUREAU_HOUR/DAY/WEEK/MON/QRT | Bỏ |

---

## 9. Flags đã tạo (giữ lại)

| Flag | Ý nghĩa | Default rate (0 vs 1) |
|---|---|---|
| DAYS_EMPLOYED_ANOM | Không đi làm chính thức | 8.66% vs 5.4% |
| EXT_SOURCE_1_MISSING | Không có điểm tín dụng nguồn 1 | 7.5% vs 8.52% |
| EXT_SOURCE_3_MISSING | Không có điểm tín dụng nguồn 3 | 7.77% vs 9.31% |
| BDS_INFO_MISSING | Không có thông tin BĐS | 6.96% vs 8.48% |
| AMT_REQ_MISSING | Không có lịch sử tra cứu tín dụng | 7.72% vs 10.34% |
| OWN_CAR_AGE_MISSING | Không có xe | 7.24% vs 8.5% |

---

## 10. Bước tiếp theo

1. **Feature Engineering** — tạo thêm feature từ các cột đã có (tỷ lệ AMT_CREDIT/AMT_INCOME, ...)
2. **Encoding** — xử lý các cột object còn lại
3. **Imputation** — xử lý missing value cho các cột còn NaN
4. **Train/Val split** — trước khi encoding để tránh data leakage
5. **Modeling** — Random Forest (giải thích) và XGBoost/LightGBM (dự đoán)

In [ ]:
bureau_bal = pd.read_csv("../Data/bureau_balance.csv").reset_index(drop=True)
bureau = pd.read_csv("../Data/bureau.csv").reset_index(drop=True)
card_bal = pd.read_csv("../Data/credit_card_balance.csv").reset_index(drop=True)
install_payments = pd.read_csv("../Data/installments_payments.csv").reset_index(
    drop=True
)
pos_cash = pd.read_csv("../Data/POS_CASH_balance.csv").reset_index(drop=True)
pre_app = pd.read_csv("../Data/previous_application.csv").reset_index(drop=True)

In [ ]:
pre_app.head()

In [ ]:
print(pre_app.shape)
print(pre_app["SK_ID_CURR"].nunique())
print(pre_app.dtypes)
print(pre_app.isnull().sum())

In [ ]:
# Aggregate theo ID CURR
pre_agg = pre_app.groupby("SK_ID_CURR", as_index=False).agg(
    # Số lần vay
    pre_loan_count=("SK_ID_PREV", "count"),
    # Tỉ lệ vay thành công/từ chối
    pre_approved_rate=("NAME_CONTRACT_STATUS", lambda x: (x == "Approved").mean()),
    pre_refused_rate=("NAME_CONTRACT_STATUS", lambda x: (x == "Refused").mean()),
    # Tổng khoản vay
    pre_amt_credit_sum=("AMT_CREDIT", "sum"),
    pre_amt_application_sum=("AMT_APPLICATION", "sum"),
    pre_amt_annuity_sum=("AMT_ANNUITY", "sum"),
    pre_amt_goodsprice_sum=("AMT_GOODS_PRICE", "sum"),
    # Trung bình khoản vay
    pre_amt_credit_mean=("AMT_CREDIT", "mean"),
    pre_amt_application_mean=("AMT_CREDIT", "mean"),
    pre_amt_annuity_mean=("AMT_ANNUITY", "mean"),
    pre_amt_goodsprice_mean=("AMT_GOODS_PRICE", "mean"),
)

In [ ]:
pre_agg.head()

In [ ]:
print(bureau.shape)
bureau.head()

In [ ]:
bureau_bal.head()

In [ ]:
# Xử lí bảng bureau_bal

# CONVERT cột STATUS qua số, các chữ sẽ chuyển thành NaN để tính max/count
status_map = {"0": 0, "1": 1, "2": 2, "3": 3, "4": 4, "5": 5, "C": np.nan, "X": np.nan}
# tạo cột STATUS mới
bureau_bal["STATUS_NUM"] = bureau_bal["STATUS"].map(status_map)

# Đếm số tháng theo từng STATUS
status_counts = pd.crosstab(bureau_bal["SK_ID_BUREAU"], bureau_bal["STATUS"])
# pivot bằng crosstab(hàng, cột(status có bn giá trị duy nhất sẽ có bn cột), mỗi ô là số lần xuất hiện)
status_counts.columns = [f"status_count_{col}" for col in status_counts.columns]
status_counts = status_counts.reset_index()

# Aggregate
bureau_bal_agg = bureau_bal.groupby("SK_ID_BUREAU", as_index=False).agg(
    months_count=("MONTHS_BALANCE", "count"),
    dpd_max=("STATUS_NUM", "max"),
    dpd_count=("STATUS_NUM", lambda x: (x >= 1).sum()),
    active_months=(
        "STATUS_NUM",
        lambda x: x.notna().sum(),
    ),  # Số tháng có status từ 0 tới 5, vì C hoặc X là không hoạt động
    dpd_severe_ever=("STATUS_NUM", lambda x: (x >= 3).any()),
)

# Gắn cờ khoản vay từng quá hạn
bureau_bal_agg["dpd_ever"] = (bureau_bal_agg["dpd_max"] >= 1).astype(int)
# Gắn cờ khoản vay quá hạn rất nặng
bureau_bal_agg["dpd_severe_ever"] = (bureau_bal_agg["dpd_severe_ever"]).astype(int)

# Tỷ lệ tháng quá hạn/số tháng hoạt động
bureau_bal_agg["dpd_ratio"] = (
    bureau_bal_agg["dpd_count"] / bureau_bal_agg["active_months"] * 100
)
# Nếu chia 0, tức là lịch sử là C hoặc X
bureau_bal_agg["dpd_ratio"] = bureau_bal_agg["dpd_ratio"].replace(
    [np.inf, -np.inf], np.nan
)

# Tháng gần nhất quá hạn (recency)
dpd_rows = bureau_bal[bureau_bal["STATUS_NUM"] >= 1]  # lọc dataset quá hạn
month_since_dpd = dpd_rows.groupby("SK_ID_BUREAU")["MONTHS_BALANCE"].max().reset_index()
month_since_dpd.columns = ["SK_ID_BUREAU", "MONTH_SINCE_LAST_DPD"]

# Status của tháng gần hiện tại nhất
lastest_idx = bureau_bal.groupby("SK_ID_BUREAU")["MONTHS_BALANCE"].idxmax()
status_last = bureau_bal.loc[lastest_idx, ["SK_ID_BUREAU", "STATUS"]].rename(
    columns={"STATUS": "STATUS_LAST"}
)

# Merge các dataset trên lại với nhau
bureau_bal_agg = bureau_bal_agg.merge(status_counts, on="SK_ID_BUREAU", how="left")
bureau_bal_agg = bureau_bal_agg.merge(month_since_dpd, on="SK_ID_BUREAU", how="left")
bureau_bal_agg = bureau_bal_agg.merge(status_last, on="SK_ID_BUREAU", how="left")

# Kiểm tra số dòng của dataset sau Aggregate
print(bureau_bal_agg.shape)
print(bureau_bal_agg.head())

In [ ]:
# Merge vào bảng bureau
bureau = bureau.merge(bureau_bal_agg, on="SK_ID_BUREAU", how="left")
print(bureau.shape)
bureau.head()
# nhìn vào bảng dưới, 1 dòng tương đương 1 khoản vay, 1 khách hàng có thể có nhiều khoản vay

In [ ]:
# Tạo cột flag_bureau_bal_missing
bureau["BUREAU_BAL_MISSING"] = bureau["months_count"].isnull().astype(int)
print(f"Tỷ lệ missing bureau_bal: {bureau["BUREAU_BAL_MISSING"].mean(): .4f}")
# tạo bảng tạm bureau và app_train để xem default rate như thế nào (vì TARGET chỉ tồn tại ở app_train)
temp = bureau[["SK_ID_CURR", "BUREAU_BAL_MISSING"]].merge(
    app_train[["SK_ID_CURR", "TARGET"]], on="SK_ID_CURR", how="inner"
)
print(f"Default Rate khi Bureau_bal_mising = 1: ")
print(temp.groupby("BUREAU_BAL_MISSING")["TARGET"].mean().round(4))

In [ ]:
print(bureau.head())
print(bureau.shape)

In [ ]:
# Xử lý nhóm cột từ bureau_bal, ví dụ như dpd_ever, dpd_serve_ever

# ===== PHẦN 1: Aggregate các cột số (numeric) từ cấp khoản vay -> cấp khách hàng =====
agg_balance_derived = (
    bureau.groupby("SK_ID_CURR")
    .agg(
        # Mức DPD (độ quá hạn) cao nhất từng đạt, tính trên TẤT CẢ khoản vay của khách hàng
        # max() vì chỉ cần biết khoản vay nào "tệ nhất" đã từng xảy ra
        bureau_dpd_max=("dpd_max", "max"),
        # dpd_ever là cờ 0/1 ở mức từng khoản vay (khoản vay này từng quá hạn chưa)
        # mean() trên cờ 0/1 sẽ cho ra TỶ LỆ % khoản vay từng quá hạn / tổng khoản vay
        # Ví dụ: 3/10 khoản vay từng quá hạn -> mean = 0.3 (30%)
        bureau_dpd_ever_rate=("dpd_ever", "mean"),
        # Tương tự dpd_ever nhưng cho mức quá hạn NẶNG (status >= 3)
        # mean() -> tỷ lệ % khoản vay từng quá hạn nặng
        bureau_dpd_severe_ever_rate=("dpd_severe_ever", "mean"),
        # dpd_ratio là tỷ lệ (số tháng quá hạn / số tháng hoạt động) TÍNH RIÊNG cho mỗi khoản vay
        # mean() ở đây là lấy trung bình của các tỷ lệ đó qua các khoản vay
        # (không phải tính lại tỷ lệ tổng, chỉ là trung bình cộng các tỷ lệ đã có)
        bureau_dpd_ratio_mean=("dpd_ratio", "mean"),
        # months_since_last_dpd: quá hạn gần nhất cách đây bao nhiêu tháng (đã convert dương ở bước trước,
        # giá trị nhỏ = gần hiện tại, NaN = chưa từng quá hạn)
        # min() để lấy ra khoản vay có lần quá hạn GẦN NHẤT trong số tất cả khoản vay
        # (vì nếu khách hàng có nhiều khoản vay, ta quan tâm lần quá hạn gần đây nhất, bất kể ở khoản nào)
        bureau_months_since_last_dpd_min=("MONTH_SINCE_LAST_DPD", "min"),
        # BUREAU_BALANCE_MISSING là cờ 0/1 (khoản vay này có thiếu lịch sử bureau_balance không)
        # mean() -> tỷ lệ % khoản vay của khách hàng KHÔNG có lịch sử bureau_balance
        bureau_balance_missing_rate=("BUREAU_BAL_MISSING", "mean"),
        # ----- Các cột bị bỏ sót ở lần trước, bổ sung lại -----
        # Tổng số tháng có dữ liệu lịch sử, CỘNG DỒN qua tất cả khoản vay của khách hàng
        # (khác với mean - ở đây muốn biết quy mô TỔNG lịch sử tín dụng, không phải trung bình mỗi khoản)
        bureau_months_count_sum=("months_count", "sum"),
        # Tổng số tháng quá hạn, CỘNG DỒN qua tất cả khoản vay
        bureau_dpd_count_sum=("dpd_count", "sum"),
        # Tổng số tháng có dữ liệu HOẠT ĐỘNG (status 0-5, không tính C/X), cộng dồn qua tất cả khoản vay
        bureau_active_months_sum=("active_months", "sum"),
        # 8 cột status_count_* (đếm số tháng ở từng trạng thái 0,1,2,3,4,5,C,X cho MỖI khoản vay)
        # Giờ cộng dồn (sum) qua tất cả khoản vay -> ra tổng số tháng khách hàng từng ở mỗi trạng thái
        # Ví dụ: bureau_status_count_3_sum = 5 -> tổng cộng 5 tháng (across mọi khoản vay) khách hàng
        # bị quá hạn ở mức 61-90 ngày
        bureau_status_count_0_sum=("status_count_0", "sum"),
        bureau_status_count_1_sum=("status_count_1", "sum"),
        bureau_status_count_2_sum=("status_count_2", "sum"),
        bureau_status_count_3_sum=("status_count_3", "sum"),
        bureau_status_count_4_sum=("status_count_4", "sum"),
        bureau_status_count_5_sum=("status_count_5", "sum"),
        bureau_status_count_C_sum=("status_count_C", "sum"),
        bureau_status_count_X_sum=("status_count_X", "sum"),
    )
    .reset_index()
)
# reset_index() để đưa SK_ID_CURR (đang là index sau groupby) trở lại thành cột thường,
# giúp merge dễ dàng ở bước sau


# ===== PHẦN 2: Xử lý riêng cột status_last (categorical, không gộp bằng sum/mean được) =====

# status_last là trạng thái của khoản vay TẠI THỜI ĐIỂM GẦN NHẤT (C, X, hoặc 0-5)
# Đây là cột dạng chữ (categorical) nên KHÔNG dùng sum/mean trực tiếp như trên được
# Cách xử lý: tính TỶ LỆ % các khoản vay của khách hàng đang ở mỗi trạng thái status_last

# pd.crosstab: đếm chéo giữa SK_ID_CURR (khách hàng) và status_last (trạng thái)
# normalize='index': chia mỗi dòng cho tổng dòng đó -> ra tỷ lệ % (tổng mỗi dòng = 1)
status_last_pivot = pd.crosstab(
    bureau["SK_ID_CURR"], bureau["STATUS_LAST"], normalize="index"
)

# Đổi tên cột để rõ nghĩa khi merge vào app_train sau này
# Ví dụ: cột 'C' -> 'bureau_status_last_rate_C' (tỷ lệ % khoản vay có status_last = Closed)
status_last_pivot.columns = [
    f"bureau_status_last_rate_{col}" for col in status_last_pivot.columns
]

# Đưa SK_ID_CURR (đang là index) trở lại thành cột thường
status_last_pivot = status_last_pivot.reset_index()


# ===== PHẦN 3: Merge 2 phần lại thành 1 bảng duy nhất theo SK_ID_CURR =====
agg_balance_derived = agg_balance_derived.merge(
    status_last_pivot, on="SK_ID_CURR", how="left"
)

# Kiểm tra kết quả
print(agg_balance_derived.shape)
print(agg_balance_derived.head())

In [ ]:
# Chuyển các cột ngày sang giá trị dương, tức là càng lớn thì càng lâu
days_bureau_cols = [
    "DAYS_CREDIT",
    "DAYS_CREDIT_ENDDATE",
    "DAYS_ENDDATE_FACT",
    "DAYS_CREDIT_UPDATE",
]
for col in days_bureau_cols:
    bureau[col] = bureau[col].abs()
    print(f"Đã chuyển giá trị từ âm sang dương cột: {col}")

In [ ]:
# Aggregate đối với nhóm CREDIT theo ID CURR
amt_bureau_agg = bureau.groupby("SK_ID_CURR", as_index=False).agg(
    # Số khoản vay của 1 khách
    bureau_count=("SK_ID_BUREAU", "count"),
    # Khoảng vay tín dụng
    bureau_amt_credit_sum_sum=("AMT_CREDIT_SUM", "sum"),
    bureau_amt_credit_sum_mean=("AMT_CREDIT_SUM", "mean"),
    # Dư nợ tín dụng
    bureau_amt_credit_debt_sum=("AMT_CREDIT_SUM_DEBT", "sum"),
    bureau_amt_credit_debt_mean=("AMT_CREDIT_SUM_DEBT", "mean"),
    # Hạn mức tín dụng
    bureau_amt_credit_limit_sum=("AMT_CREDIT_SUM_LIMIT", "sum"),
    bureau_amt_credit_limit_mean=("AMT_CREDIT_SUM_LIMIT", "mean"),
    # Giá trị thanh toán định kì các khoản vay
    bureau_amt_annuity_sum=("AMT_ANNUITY", "sum"),
    bureau_amt_annuity_mean=("AMT_ANNUITY", "mean"),
)

In [ ]:
# Xử lý NaN trong bureau_amt_annuity_mean
# Giữ NaN - để model tự xử lý (XGBoost/LightGBM), không impute vội
# Vì NaN ở đây mang nghĩa "không có khoản vay nào có annuity"
# khác với 0 (có annuity nhưng giá trị = 0)
print(amt_bureau_agg.head())

In [ ]:
# Xử lý nhóm CREDIT ACTIVE
credit_active_pivot = pd.crosstab(
    bureau["SK_ID_CURR"], bureau["CREDIT_ACTIVE"], normalize="index"
)
credit_active_pivot.columns = [
    f"bureau_active_rate_{col.lower()}" for col in credit_active_pivot.columns
]
rate_cols = [
    col for col in credit_active_pivot.columns if col.startswith("bureau_active_rate_")
]
credit_active_pivot[rate_cols] = (credit_active_pivot[rate_cols] * 100).round(2)
credit_active_pivot = credit_active_pivot.reset_index()

print(credit_active_pivot.head())
print(bureau["CREDIT_ACTIVE"].value_counts())

In [ ]:
print(bureau["CREDIT_TYPE"].value_counts().reset_index())

# Gộp các khoản vay ít xuất hiện lại với nhau thành Other
top_credit_types = [
    "Consumer credit",
    "Credit card",
    "Car loan",
    "Mortgage",
    "Microloan",
]
bureau["CREDIT_TYPE_GROUPED"] = bureau["CREDIT_TYPE"].apply(
    lambda x: x if x in top_credit_types else "Other"
)

# Xử lý nhóm CREDIT TYPE
credit_type_pivot = pd.crosstab(
    bureau["SK_ID_CURR"], bureau["CREDIT_TYPE_GROUPED"], normalize="index"
)
credit_type_pivot.columns = [
    f"bureau_type_rate_{col.lower().replace(" ", "_")}"
    for col in credit_type_pivot.columns
]
rate_cols = [
    col for col in credit_type_pivot.columns if col.startswith("bureau_type_rate_")
]
credit_type_pivot[rate_cols] = (credit_type_pivot[rate_cols] * 100).round(2)
# Mỗi khách sẽ có bao nhiêu khoản vay
bureau_count = bureau.groupby("SK_ID_CURR").size().reset_index(name="bureau_count")
credit_type_pivot = credit_type_pivot.reset_index()
credit_type_pivot = credit_type_pivot.merge(bureau_count, on="SK_ID_CURR", how="left")

print(credit_type_pivot.head())

In [ ]:
# Xử lí nhóm cột DAYS

days_pre_cols = [
    "DAYS_FIRST_DRAWING",
    "DAYS_FIRST_DUE",
    "DAYS_LAST_DUE_1ST_VERSION",
    "DAYS_LAST_DUE",
    "DAYS_TERMINATION",
]

for col in days_pre_cols:
    pre_app[col] = pre_app[col].replace(365243, np.nan)
    print(f"Đã xử lý cột: {col}")

bureau_day_agg = bureau.groupby("SK_ID_CURR", as_index=False).agg(
    # Khoản vay mới nhất và cũ nhất
    bureau_day_credit_min=("DAYS_CREDIT", "min"),  # lần vay mới nhất
    bureau_day_credit_max=("DAYS_CREDIT", "max"),  # lần vay cũ nhất
    bureau_day_credit_mean=("DAYS_CREDIT", "mean"),
    # Thời hạn còn lại trung bình của khoản vay
    bureau_days_credit_enddate_mean=("DAYS_CREDIT_ENDDATE", "mean"),
    # Khoản vay cuối đóng cách đây bao lâu (khoản Closed)
    bureau_day_enddate_fact_min=("DAYS_ENDDATE_FACT", "min"),  # khoản vay đã đóng
    # Lần cập nhật cuối
    bureau_day_credit_update_min=("DAYS_CREDIT_UPDATE", "min"),
)
print("=" * 50)
print(bureau_day_agg.head())
print(bureau_day_agg.describe())
print("Sau khi chạy, phát hiện các cột DAYS_CREDIT_* có outlier làm lệch dữ liệu!")

In [ ]:
# Chuẩn hóa lại các cột có outlier
print("Phân phối của các cột enddate và update: ")
print(bureau["DAYS_CREDIT_ENDDATE"].describe())
print(bureau["DAYS_CREDIT_UPDATE"].describe())
print("=" * 50)

# Đếm số dòng outlier
print("Số lượng khoản vay mà thời gian kết thúc và update trên 20.000 ~ 54,8 năm: ")
print((bureau["DAYS_CREDIT_ENDDATE"] > 20000).sum())
print((bureau["DAYS_CREDIT_UPDATE"] > 20000).sum())
print("=" * 50)

# Phân phối của nhóm > 20000
print("Lọc dataset của dữ liệu trên 20.000 xem: ")
print(bureau[bureau["DAYS_CREDIT_ENDDATE"] > 20000]["CREDIT_TYPE"].value_counts())
print(
    "Kết quả thể hiện 2 khoản vay tiêu dùng và thẻ tín dụng lại có kì hạn ~55 năm trở lên, vô lý!!!"
)
print("=" * 50)
print("Phân bố các giá trị dưới 20.000")
print(bureau[bureau["DAYS_CREDIT_ENDDATE"] <= 20000]["DAYS_CREDIT_ENDDATE"].describe())

# Thay thế các giá trị trên 20.000 thành Nan
bureau.loc[bureau["DAYS_CREDIT_ENDDATE"] > 20000, "DAYS_CREDIT_ENDDATE"] = np.nan
bureau.loc[bureau["DAYS_CREDIT_UPDATE"] > 20000, "DAYS_CREDIT_UPDATE"] = np.nan
print("Phân phối của các cột enddate và update sau khi thay thế Nan: ")
print(bureau["DAYS_CREDIT_ENDDATE"].describe())
print(bureau["DAYS_CREDIT_UPDATE"].describe())

In [ ]:
print("Chạy lại bureau_day_agg để cho kết quả chuẩn hơn: ")

bureau_day_agg = bureau.groupby("SK_ID_CURR", as_index=False).agg(
    # Khoản vay mới nhất và cũ nhất
    bureau_day_credit_min=("DAYS_CREDIT", "min"),  # lần vay mới nhất
    bureau_day_credit_max=("DAYS_CREDIT", "max"),  # lần vay cũ nhất
    bureau_day_credit_mean=("DAYS_CREDIT", "mean"),
    # Thời hạn còn lại trung bình của khoản vay
    bureau_days_credit_enddate_mean=("DAYS_CREDIT_ENDDATE", "mean"),
    # Khoản vay cuối đóng cách đây bao lâu (khoản Closed)
    bureau_day_enddate_fact_min=("DAYS_ENDDATE_FACT", "min"),  # khoản vay đã đóng
    # Lần cập nhật cuối
    bureau_day_credit_update_min=("DAYS_CREDIT_UPDATE", "min"),
)
print("=" * 50)
print(bureau_day_agg.head())
print("=" * 50)
print(bureau_day_agg.describe())

In [ ]:
# Xem nhóm Overdue
bureau_overdue_agg = bureau.groupby("SK_ID_CURR", as_index=False).agg(
    # Số ngày quá hạn cao nhất từng có (trong số tất cả khoản vay của khách hàng)
    # max() vì chỉ cần biết mức độ nghiêm trọng nhất, không cần trung bình
    bureau_credit_day_overdue_max=("CREDIT_DAY_OVERDUE", "max"),
    # Số tiền quá hạn cao nhất từng ghi nhận (trong lịch sử, kể cả đã trả)
    # Đây là "vết" quá hạn nặng nhất từng có, dù hiện tại đã hết quá hạn
    bureau_amt_credit_max_overdue=("AMT_CREDIT_MAX_OVERDUE", "max"),
    # Tổng số tiền đang quá hạn HIỆN TẠI, cộng dồn tất cả khoản vay
    # sum() vì muốn biết tổng gánh nợ quá hạn của khách hàng lúc này
    bureau_amt_credit_sum_overdue_sum=("AMT_CREDIT_SUM_OVERDUE", "sum"),
    # Số tiền quá hạn HIỆN TẠI cao nhất trong 1 khoản vay cụ thể
    # max() để biết khoản vay nào đang "nặng" nhất
    bureau_amt_credit_sum_overdue_max=("AMT_CREDIT_SUM_OVERDUE", "max"),
    # Tổng số lần khách hàng được gia hạn khoản vay (cộng dồn tất cả khoản vay)
    # sum() vì muốn biết tổng số lần "xin gia hạn" nợ
    bureau_cnt_credit_prolong_sum=("CNT_CREDIT_PROLONG", "sum"),
)

print(bureau_overdue_agg.describe())

In [ ]:
# Kiểm tra phân phối tổng thể của cột AMT_CREDIT_MAX_OVERDUE
# -> Phát hiện max quá lớn (115,987,185) so với phần lớn còn lại (75% vẫn = 0)
print("Phát hiện credit_max_overdue có outlier nên xem lại phân phối: ")
print(bureau["AMT_CREDIT_MAX_OVERDUE"].describe())

# Xem chi tiết các dòng có giá trị > 10 triệu để hiểu rõ bản chất outlier này
# (kiểm tra xem chúng thuộc loại khoản vay nào, có hợp lý không)
print(bureau[bureau["AMT_CREDIT_MAX_OVERDUE"] > 10000000])

# In ra các ngưỡng phân vị từ 95% đến 99.99%
# Mục đích: xem các giá trị >10 triệu nằm ở đâu trong phân phối
# Nếu chúng vẫn nằm trong nhóm 99.99% thông thường -> có thể là giá trị thực, chỉ là hiếm
# Nếu chúng vượt xa cả mốc 99.99% -> là outlier cực đoan, cần xử lý
print("=" * 50)
print(bureau["AMT_CREDIT_MAX_OVERDUE"].quantile([0.95, 0.99, 0.999, 0.9999]))

# Kết quả: các giá trị >10 triệu vượt xa cả mốc 99.99% (2,122,442)
# -> Đây là outlier cực đoan, không phải biến động tự nhiên của phân phối
# -> Hướng xử lý: dùng clip() để giới hạn giá trị tối đa,
#    thay vì xóa dòng hoặc gán NaN (vì vẫn có thể là giá trị thực, chỉ cần giới hạn lại biên trên)

In [ ]:
# Clip dữ liệu về biên trên là giá trị tại ngưỡng 99,99%
cap_value = bureau["AMT_CREDIT_MAX_OVERDUE"].quantile(0.9999)
bureau["AMT_CREDIT_MAX_OVERDUE"] = bureau["AMT_CREDIT_MAX_OVERDUE"].clip(
    upper=cap_value
)
print(bureau["AMT_CREDIT_MAX_OVERDUE"].describe())
print("=" * 50)
print("Chạy lại biến: bureau_overdue_agg: ")

# Xem nhóm Overdue
bureau_overdue_agg = bureau.groupby("SK_ID_CURR", as_index=False).agg(
    # Số ngày quá hạn cao nhất từng có (trong số tất cả khoản vay của khách hàng)
    # max() vì chỉ cần biết mức độ nghiêm trọng nhất, không cần trung bình
    bureau_credit_day_overdue_max=("CREDIT_DAY_OVERDUE", "max"),
    # Số tiền quá hạn cao nhất từng ghi nhận (trong lịch sử, kể cả đã trả)
    # Đây là "vết" quá hạn nặng nhất từng có, dù hiện tại đã hết quá hạn
    bureau_amt_credit_max_overdue=("AMT_CREDIT_MAX_OVERDUE", "max"),
    # Tổng số tiền đang quá hạn HIỆN TẠI, cộng dồn tất cả khoản vay
    # sum() vì muốn biết tổng gánh nợ quá hạn của khách hàng lúc này
    bureau_amt_credit_sum_overdue_sum=("AMT_CREDIT_SUM_OVERDUE", "sum"),
    # Số tiền quá hạn HIỆN TẠI cao nhất trong 1 khoản vay cụ thể
    # max() để biết khoản vay nào đang "nặng" nhất
    bureau_amt_credit_sum_overdue_max=("AMT_CREDIT_SUM_OVERDUE", "max"),
    # Tổng số lần khách hàng được gia hạn khoản vay (cộng dồn tất cả khoản vay)
    # sum() vì muốn biết tổng số lần "xin gia hạn" nợ
    bureau_cnt_credit_prolong_sum=("CNT_CREDIT_PROLONG", "sum"),
)

print(bureau_overdue_agg.describe())

In [ ]:
final_bureau = (
    amt_bureau_agg.merge(credit_active_pivot, on="SK_ID_CURR", how="outer")
    .merge(credit_type_pivot, on="SK_ID_CURR", how="outer")
    .merge(bureau_day_agg, on="SK_ID_CURR", how="outer")
    .merge(bureau_overdue_agg, on="SK_ID_CURR", how="outer")
    .merge(agg_balance_derived, on="SK_ID_CURR", how="outer")
)

print(f"Số hàng: {final_bureau.shape[0]}")
print(f"Số cột: {final_bureau.shape[1]}")
print(final_bureau.head())

In [ ]:
# Kiểm tra 2 cột có giống nhau không
print("Phát hiện bị trùng cột bureau_count nên xóa bớt 1: ")
print(final_bureau[["bureau_count_x", "bureau_count_y"]].head(10))
print((final_bureau["bureau_count_x"] == final_bureau["bureau_count_y"]).all())
final_bureau = final_bureau.drop(columns=["bureau_count_x"])
final_bureau = final_bureau.rename(columns={"bureau_count_y": "bureau_count"})

In [ ]:
print(f"app_train trước khi merge: {app_train.shape}")
print(f"app_test trước khi merge: {app_test.shape}")

print("Merge final_bureau, pre_app vào app_train và app_test: ")
app_train = app_train.merge(final_bureau, on="SK_ID_CURR", how="left")
app_train = app_train.merge(pre_agg, on="SK_ID_CURR", how="left")

app_test = app_test.merge(final_bureau, on="SK_ID_CURR", how="left")
app_test = app_test.merge(pre_agg, on="SK_ID_CURR", how="left")

print(f"app_train sau khi merge: {app_train.shape}")
print(f"app_test sau khi merge: {app_test.shape}")

# Tóm tắt xử lý bảng phụ: previous_application, bureau, bureau_balance

## Tổng quan pipeline
bureau_balance (27,299,925 dòng)

↓ aggregate theo SK_ID_BUREAU

bureau_bal_agg (1 dòng/khoản vay)

↓ merge vào bureau theo SK_ID_BUREAU

bureau mới (1,716,428 dòng, 35 cột)

↓ aggregate theo SK_ID_CURR → 6 bảng agg

bureau_agg (305,811 dòng, 56 cột) — 1 dòng/khách hàng

↓ merge vào app_train/app_test
previous_application (1,670,214 dòng, 37 cột)

↓ aggregate theo SK_ID_CURR

prev_agg (1 dòng/khách hàng)

↓ merge vào app_train/app_test

---

## 1. bureau_balance

### Đặc điểm
- Mỗi dòng = 1 tháng lịch sử của 1 khoản vay bureau
- Join key: `SK_ID_BUREAU`
- Cột quan trọng: `MONTHS_BALANCE` (tháng, âm = quá khứ), `STATUS` (C/X/0-5)

### Xử lý STATUS
| Giá trị | Ý nghĩa |
|---|---|
| 0 | Không quá hạn |
| 1-5 | Quá hạn tăng dần (1=1-30 ngày, 5=120+ ngày hoặc nợ xấu) |
| C | Closed — đã đóng |
| X | Unknown — không có thông tin |

→ Map C và X thành NaN (cột `STATUS_NUM`) để tính toán DPD, giữ nguyên cho `status_last`

### Features tạo ra (aggregate theo SK_ID_BUREAU)
| Feature | Ý nghĩa | Hàm |
|---|---|---|
| `months_count` | Độ dài lịch sử (số tháng có dữ liệu) | `count` |
| `dpd_max` | Mức DPD cao nhất từng đạt | `max` |
| `dpd_count` | Số tháng từng quá hạn | `lambda x: (x>=1).sum()` |
| `active_months` | Số tháng có dữ liệu hoạt động (status 0-5) | `count` trên STATUS_NUM |
| `dpd_ratio` | Tỷ lệ tháng quá hạn / tháng hoạt động (%) | `dpd_count / active_months * 100` |
| `dpd_ever` | Từng quá hạn chưa (binary) | `dpd_max >= 1` |
| `dpd_severe_ever` | Từng quá hạn nặng (status >= 3) | `(x>=3).any()` |
| `months_since_last_dpd` | Tháng gần nhất bị quá hạn | `max` của MONTHS_BALANCE khi STATUS_NUM >= 1 |
| `status_count_0..5,C,X` | Số tháng ở từng trạng thái | `pd.crosstab` |
| `status_last` | Trạng thái tháng gần nhất | `idxmax` của MONTHS_BALANCE |
| `BUREAU_BAL_MISSING` | Khoản vay không có lịch sử balance | flag 0/1 |

---

## 2. bureau

### Đặc điểm
- Mỗi dòng = 1 khoản vay tại Credit Bureau của khách hàng
- Join key: `SK_ID_BUREAU` (với bureau_balance), `SK_ID_CURR` (với app_train)
- 1 khách hàng có thể có nhiều khoản vay bureau

### Xử lý đặc biệt
- `DAYS_CREDIT_ENDDATE > 20,000` (~38,588 dòng, toàn Credit card/Consumer credit) → thay NaN (lỗi nhập liệu, không thể có thời hạn 55+ năm)
- `DAYS_CREDIT_UPDATE > 20,000` (95 dòng) → thay NaN
- `AMT_CREDIT_MAX_OVERDUE > 99.99th percentile` → clip (outlier cực đoan nhưng có thể là giá trị thực)
- Tất cả cột DAYS → convert sang số dương, đơn vị ngày
- `CREDIT_TYPE` → gộp thành 6 nhóm: Consumer credit, Credit card, Car loan, Mortgage, Microloan, Other

### Pipeline aggregate (6 bảng → merge thành bureau_agg)

#### Nhóm 1 — Số tiền (amt_bureau_agg)
| Feature | Ý nghĩa | Hàm |
|---|---|---|
| `bureau_amt_credit_sum_sum/mean` | Tổng/trung bình giá trị khoản vay | `sum`, `mean` |
| `bureau_amt_credit_debt_sum/mean` | Tổng/trung bình dư nợ hiện tại | `sum`, `mean` |
| `bureau_amt_credit_limit_sum/mean` | Tổng/trung bình hạn mức (thẻ tín dụng) | `sum`, `mean` |
| `bureau_amt_annuity_sum/mean` | Tổng/trung bình khoản trả góp | `sum`, `mean` |

#### Nhóm 2 — Trạng thái khoản vay (credit_active_pivot)
- Tỷ lệ % khoản vay theo `CREDIT_ACTIVE`: Active, Closed, Sold, Bad debt
- `bureau_count` — tổng số khoản vay bureau

#### Nhóm 3 — Loại khoản vay (credit_type_pivot)
- Tỷ lệ % khoản vay theo `CREDIT_TYPE_GROUPED`: 6 loại

#### Nhóm 4 — Thời gian (bureau_day_agg)
| Feature | Ý nghĩa | Hàm |
|---|---|---|
| `bureau_days_credit_min/max/mean` | Khoản vay gần nhất/đầu tiên/trung bình | `min`, `max`, `mean` |
| `bureau_days_credit_enddate_mean` | Thời hạn còn lại trung bình | `mean` |
| `bureau_days_enddate_fact_min` | Khoản vay đóng gần nhất | `min` |
| `bureau_days_credit_update_min` | Lần cập nhật gần nhất | `min` |

#### Nhóm 5 — Quá hạn hiện tại (bureau_overdue_agg)
| Feature | Ý nghĩa | Hàm |
|---|---|---|
| `bureau_credit_day_overdue_max` | Số ngày quá hạn cao nhất hiện tại | `max` |
| `bureau_amt_credit_max_overdue` | Số tiền quá hạn cao nhất từng có | `max` |
| `bureau_amt_credit_sum_overdue_sum/max` | Tổng/max số tiền đang quá hạn | `sum`, `max` |
| `bureau_cnt_credit_prolong_sum` | Tổng số lần gia hạn | `sum` |

#### Nhóm 6 — Derived từ bureau_balance (agg_balance_derived)
| Feature | Ý nghĩa | Hàm |
|---|---|---|
| `bureau_dpd_max` | Mức DPD cao nhất across tất cả khoản vay | `max` |
| `bureau_dpd_ever_rate` | Tỷ lệ % khoản vay từng quá hạn | `mean` |
| `bureau_dpd_severe_ever_rate` | Tỷ lệ % khoản vay từng quá hạn nặng | `mean` |
| `bureau_dpd_ratio_mean` | Trung bình tỷ lệ tháng quá hạn (%) | `mean` |
| `bureau_months_since_last_dpd_min` | Quá hạn gần nhất (tháng) | `min` |
| `bureau_balance_missing_rate` | Tỷ lệ % khoản vay thiếu lịch sử balance | `mean` |
| `bureau_months_count_sum` | Tổng số tháng lịch sử | `sum` |
| `bureau_dpd_count_sum` | Tổng số tháng quá hạn | `sum` |
| `bureau_active_months_sum` | Tổng số tháng hoạt động | `sum` |
| `bureau_status_count_0..5,C,X_sum` | Tổng số tháng ở từng trạng thái | `sum` |
| `bureau_status_last_rate_0..5,C,X` | Tỷ lệ % khoản vay có status_last = mỗi trạng thái | `crosstab normalize` |

---

## 3. previous_application

### Đặc điểm
- Mỗi dòng = 1 đơn vay trước đây của khách hàng tại Home Credit
- Join key: `SK_ID_CURR`
- 1,670,214 dòng, 338,857 khách hàng unique (~4.9 đơn/khách hàng)
- `NAME_CONTRACT_STATUS` có 4 giá trị: Approved, Refused, Canceled, Unused offer

### Xử lý đặc biệt
- Các cột `DAYS_*` có giá trị 365,243 (placeholder) → thay NaN trước khi aggregate

### Features tạo ra (aggregate theo SK_ID_CURR → prev_agg)
| Feature | Ý nghĩa | Hàm |
|---|---|---|
| `prev_loan_count` | Tổng số đơn vay trước đây | `count` |
| `prev_approved_rate` | Tỷ lệ đơn được duyệt | `lambda x: (x=='Approved').mean()` |
| `prev_refused_rate` | Tỷ lệ đơn bị từ chối | `lambda x: (x=='Refused').mean()` |
| `prev_amt_credit_mean/sum` | Trung bình/tổng số tiền vay trước | `mean`, `sum` |
| `prev_amt_application_mean` | Trung bình số tiền xin vay | `mean` |
| `prev_amt_credit_app_ratio` | Tỷ lệ credit/application (được duyệt so với xin) | `sum/sum` |
| `prev_days_decision_min/mean` | Đơn gần nhất/trung bình cách đây bao lâu | `min`, `mean` |

---

## 4. Kết quả sau merge vào app_train/app_test

| | Trước merge | Sau merge |
|---|---|---|
| app_train | (307,511, 128) | (307,511, 194) |
| app_test | (48,744, 121) | (48,744, 187) |

**Lưu ý:** Khách hàng không có trong bureau hoặc previous_application sẽ có NaN ở các cột tương ứng — xử lý ở bước imputation sau.

In [ ]:
print("card_bal columns:", card_bal.columns.tolist())
print("install_payments columns:", install_payments.columns.tolist())
print("pos_cash columns:", pos_cash.columns.tolist())

In [ ]:
print(card_bal.shape)
print(card_bal.isnull().sum())
card_bal.describe()

In [ ]:
# Xử lí missing value
# Impute 0 cho các cột giao dịch
cols_fill_zero = [
    "AMT_DRAWINGS_ATM_CURRENT",
    "AMT_DRAWINGS_OTHER_CURRENT",
    "AMT_DRAWINGS_POS_CURRENT",
    "CNT_DRAWINGS_ATM_CURRENT",
    "CNT_DRAWINGS_OTHER_CURRENT",
    "CNT_DRAWINGS_POS_CURRENT",
    "AMT_PAYMENT_CURRENT",
]
card_bal[cols_fill_zero] = card_bal[cols_fill_zero].fillna(0)

In [ ]:
# Xem outlier của SK_DPD và SK_DPD_DEF
print(card_bal["SK_DPD"].quantile([0.95, 0.99, 0.999, 0.9999]))
print(card_bal["SK_DPD_DEF"].quantile([0.95, 0.99, 0.999, 0.9999]))

# Clip dữ liệu về biên trên là giá trị tại ngưỡng 99,9%
dpd_cap = card_bal["SK_DPD"].quantile(0.999)
card_bal["SK_DPD"] = card_bal["SK_DPD"].clip(upper=dpd_cap)

dpd_def_cap = card_bal["SK_DPD_DEF"].quantile(0.999)
card_bal["SK_DPD_DEF"] = card_bal["SK_DPD"].clip(upper=dpd_def_cap)

print(card_bal["SK_DPD"].describe())
print(card_bal["SK_DPD_DEF"].describe())

In [ ]:
# Giờ Agg để merge vào bảng train/test
card_bal_agg = card_bal.groupby("SK_ID_CURR", as_index=False).agg(
    # ===== LỊCH SỬ =====
    # Tổng số tháng có dữ liệu lịch sử thẻ tín dụng
    # Khách hàng dùng thẻ lâu năm thường có lịch sử tín dụng tốt hơn
    card_months_count=("MONTHS_BALANCE", "count"),
    # ===== SỐ DƯ VÀ HẠN MỨC =====
    # Số dư trung bình hàng tháng — phản ánh mức độ sử dụng thẻ thường xuyên
    card_amt_balance_mean=("AMT_BALANCE", "mean"),
    # Số dư cao nhất từng có — phản ánh mức độ vay tối đa
    card_amt_balance_max=("AMT_BALANCE", "max"),
    # Hạn mức tín dụng trung bình — khách hàng được tin tưởng cao thường có hạn mức lớn hơn
    card_amt_credit_limit_mean=("AMT_CREDIT_LIMIT_ACTUAL", "mean"),
    # ===== GIAO DỊCH =====
    # Tổng số tiền rút/giao dịch trung bình hàng tháng
    # Người dùng thẻ nhiều không hẳn là rủi ro, nhưng kết hợp với DPD thì có ý nghĩa
    card_amt_drawings_mean=("AMT_DRAWINGS_CURRENT", "mean"),
    # Số tiền thanh toán trung bình hàng tháng
    # Người thanh toán đều đặn -> ít rủi ro hơn
    card_amt_payment_mean=("AMT_PAYMENT_CURRENT", "mean"),
    # Tổng số tiền thanh toán (bao gồm cả phí, lãi...)
    card_amt_payment_total_mean=("AMT_PAYMENT_TOTAL_CURRENT", "mean"),
    # ===== DPD (NGÀY QUÁ HẠN) =====
    # Số ngày quá hạn cao nhất từng có — signal mạnh nhất về rủi ro
    card_dpd_max=("SK_DPD", "max"),
    # Số ngày quá hạn trung bình — phản ánh mức độ quá hạn "thường xuyên"
    card_dpd_mean=("SK_DPD", "mean"),
    # Số ngày quá hạn có tolerance cao nhất (bỏ qua các khoản nợ nhỏ)
    # Thường đáng tin cậy hơn SK_DPD vì loại bỏ noise từ các khoản nhỏ
    card_dpd_def_max=("SK_DPD_DEF", "max"),
    # Số tháng từng bị quá hạn (SK_DPD > 0)
    # Khác với dpd_max (mức độ nặng nhất), cái này đo TẦN SUẤT quá hạn
    card_dpd_ever_count=("SK_DPD", lambda x: (x > 0).sum()),
)

# ===== TỶ LỆ SỬ DỤNG HẠN MỨC (UTILIZATION) =====
# Tính riêng vì cần xử lý trường hợp limit = 0 (tránh chia cho 0)
# Utilization = balance / credit_limit
# Feature RẤT QUAN TRỌNG trong credit scoring:
# - Utilization cao (>80%) -> đang dùng gần hết hạn mức -> rủi ro cao
# - Utilization thấp (<30%) -> dùng thẻ có kiểm soát -> ít rủi ro
# replace(0, np.nan) để tránh chia cho 0, kết quả NaN sẽ bị bỏ qua khi tính mean
card_bal["utilization"] = card_bal["AMT_BALANCE"] / card_bal[
    "AMT_CREDIT_LIMIT_ACTUAL"
].replace(0, np.nan)

# Aggregate utilization theo khách hàng
util_agg = card_bal.groupby("SK_ID_CURR", as_index=False).agg(
    # Tỷ lệ sử dụng hạn mức trung bình qua các tháng
    card_utilization_mean=("utilization", "mean"),
    # Tỷ lệ sử dụng hạn mức cao nhất từng có
    # Utilization max cao -> từng "kẹt tiền" ở mức nghiêm trọng
    card_utilization_max=("utilization", "max"),
)

# Merge utilization vào card_bal_agg
card_bal_agg = card_bal_agg.merge(util_agg, on="SK_ID_CURR", how="left")

print(f"Shape card_bal_agg: {card_bal_agg.shape}")
print(card_bal_agg.head())

In [ ]:
print(card_bal["utilization"].describe())
print(card_bal["utilization"].quantile([0.99, 0.999, 0.9999]))

In [ ]:
# Kiểm tra các dòng có utilization cực đoan
print(
    card_bal[card_bal["utilization"] > 5][
        ["AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL", "utilization"]
    ]
)
print(
    card_bal[card_bal["utilization"] < -1][
        ["AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL", "utilization"]
    ]
)
print("Clip utilization trong khoảng [-2, 2]")
# -2: trả dư tối đa 200% hạn mức — đủ rộng để giữ các trường hợp hợp lý
#  2: dùng vượt hạn mức tối đa 200% — đủ rộng nhưng loại bỏ dòng 11.78x bất thường
print("Chạy lại card_bal_agg, sau khi clip: ")

In [ ]:
# Giờ Agg để merge vào bảng train/test
card_bal_agg = card_bal.groupby("SK_ID_CURR", as_index=False).agg(
    # ===== LỊCH SỬ =====
    # Tổng số tháng có dữ liệu lịch sử thẻ tín dụng
    # Khách hàng dùng thẻ lâu năm thường có lịch sử tín dụng tốt hơn
    card_months_count=("MONTHS_BALANCE", "count"),
    # ===== SỐ DƯ VÀ HẠN MỨC =====
    # Số dư trung bình hàng tháng — phản ánh mức độ sử dụng thẻ thường xuyên
    card_amt_balance_mean=("AMT_BALANCE", "mean"),
    # Số dư cao nhất từng có — phản ánh mức độ vay tối đa
    card_amt_balance_max=("AMT_BALANCE", "max"),
    # Hạn mức tín dụng trung bình — khách hàng được tin tưởng cao thường có hạn mức lớn hơn
    card_amt_credit_limit_mean=("AMT_CREDIT_LIMIT_ACTUAL", "mean"),
    # ===== GIAO DỊCH =====
    # Tổng số tiền rút/giao dịch trung bình hàng tháng
    # Người dùng thẻ nhiều không hẳn là rủi ro, nhưng kết hợp với DPD thì có ý nghĩa
    card_amt_drawings_mean=("AMT_DRAWINGS_CURRENT", "mean"),
    # Số tiền thanh toán trung bình hàng tháng
    # Người thanh toán đều đặn -> ít rủi ro hơn
    card_amt_payment_mean=("AMT_PAYMENT_CURRENT", "mean"),
    # Tổng số tiền thanh toán (bao gồm cả phí, lãi...)
    card_amt_payment_total_mean=("AMT_PAYMENT_TOTAL_CURRENT", "mean"),
    # ===== DPD (NGÀY QUÁ HẠN) =====
    # Số ngày quá hạn cao nhất từng có — signal mạnh nhất về rủi ro
    card_dpd_max=("SK_DPD", "max"),
    # Số ngày quá hạn trung bình — phản ánh mức độ quá hạn "thường xuyên"
    card_dpd_mean=("SK_DPD", "mean"),
    # Số ngày quá hạn có tolerance cao nhất (bỏ qua các khoản nợ nhỏ)
    # Thường đáng tin cậy hơn SK_DPD vì loại bỏ noise từ các khoản nhỏ
    card_dpd_def_max=("SK_DPD_DEF", "max"),
    # Số tháng từng bị quá hạn (SK_DPD > 0)
    # Khác với dpd_max (mức độ nặng nhất), cái này đo TẦN SUẤT quá hạn
    card_dpd_ever_count=("SK_DPD", lambda x: (x > 0).sum()),
)

# ===== TỶ LỆ SỬ DỤNG HẠN MỨC (UTILIZATION) =====
# Tính riêng vì cần xử lý trường hợp limit = 0 (tránh chia cho 0)
# Utilization = balance / credit_limit
# Feature RẤT QUAN TRỌNG trong credit scoring:
# - Utilization cao (>80%) -> đang dùng gần hết hạn mức -> rủi ro cao
# - Utilization thấp (<30%) -> dùng thẻ có kiểm soát -> ít rủi ro
# replace(0, np.nan) để tránh chia cho 0, kết quả NaN sẽ bị bỏ qua khi tính mean
card_bal["utilization"] = card_bal["AMT_BALANCE"] / card_bal[
    "AMT_CREDIT_LIMIT_ACTUAL"
].replace(0, np.nan)
# -2: trả dư tối đa 200% hạn mức — đủ rộng để giữ các trường hợp hợp lý
#  2: dùng vượt hạn mức tối đa 200% — đủ rộng nhưng loại bỏ dòng 11.78x bất thường
card_bal["utilization"] = card_bal["utilization"].clip(lower=-2, upper=2)

# Aggregate utilization theo khách hàng
util_agg = card_bal.groupby("SK_ID_CURR", as_index=False).agg(
    # Tỷ lệ sử dụng hạn mức trung bình qua các tháng
    card_utilization_mean=("utilization", "mean"),
    # Tỷ lệ sử dụng hạn mức cao nhất từng có
    # Utilization max cao -> từng "kẹt tiền" ở mức nghiêm trọng
    card_utilization_max=("utilization", "max"),
)

# Merge utilization vào card_bal_agg
card_bal_agg = card_bal_agg.merge(util_agg, on="SK_ID_CURR", how="left")

print(f"Shape card_bal_agg: {card_bal_agg.shape}")
print(card_bal_agg.head())

In [ ]:
print(pos_cash.shape)
print(pos_cash.isnull().sum())
print(pos_cash.describe())

In [ ]:
# Không xử lí missing
# Xem outlier của SK_DPD và SK_DPD_DEF
print(pos_cash["SK_DPD"].quantile([0.95, 0.99, 0.999, 0.9999]))
print(pos_cash["SK_DPD_DEF"].quantile([0.95, 0.99, 0.999, 0.9999]))

# Clip dữ liệu về biên trên là giá trị tại ngưỡng 99,9%
dpd_cap = pos_cash["SK_DPD"].quantile(0.999)
pos_cash["SK_DPD"] = pos_cash["SK_DPD"].clip(upper=dpd_cap)

dpd_def_cap = pos_cash["SK_DPD_DEF"].quantile(0.999)
pos_cash["SK_DPD_DEF"] = pos_cash["SK_DPD_DEF"].clip(upper=dpd_def_cap)

print(pos_cash["SK_DPD"].describe())
print(pos_cash["SK_DPD_DEF"].describe())

In [ ]:
print(pos_cash["NAME_CONTRACT_STATUS"].value_counts().reset_index())
# Nhóm các trạng thái lại
status_map = {
    "Active": "Active",
    "Completed": "Completed",
    "Signed": "Pending",  # Đã ký nhưng chưa giải ngân -> chờ
    "Approved": "Pending",  # Đã duyệt nhưng chưa giải ngân -> chờ
    "Demand": "High_risk",  # Nợ xấu tiềm năng
    "Amortized debt": "High_risk",  # Nợ đã khấu hao nhưng rủi ro cao
    "Returned to the store": "Canceled",  # Hàng trả lại -> Hủy
    "Canceled": "Canceled",
    "XNA": "Pending",  # Không có thông tin và chỉ có 2 dòng nên gộp vào Pending
}

pos_cash["CONTRACT_STATUS_GROUPED"] = pos_cash["NAME_CONTRACT_STATUS"].map(status_map)
print(pos_cash["CONTRACT_STATUS_GROUPED"].value_counts())

In [ ]:
pos_cash["POS_HIGHRISK_FLAG"] = (
    pos_cash["CONTRACT_STATUS_GROUPED"] == "High_risk"
).astype(int)

pos_cash["POS_COMPLETED_FLAG"] = (
    pos_cash["CONTRACT_STATUS_GROUPED"] == "Completed"
).astype(int)

print(f"\n Số dòng high risk: {pos_cash["POS_HIGHRISK_FLAG"].sum()}")
print(f"\n Số dòng completed: {pos_cash["POS_COMPLETED_FLAG"].sum()}")

In [ ]:
pos_cash_agg = pos_cash.groupby("SK_ID_CURR", as_index=False).agg(
    # ===== ĐỘ DÀI LỊCH SỬ =====
    # Tổng số tháng có dữ liệu POS/Cash loan
    # Khách hàng có lịch sử dài hơn → ngân hàng có nhiều thông tin hơn để đánh giá
    pos_months_count=("MONTHS_BALANCE", "count"),
    # ===== DPD (NGÀY QUÁ HẠN) =====
    # Số ngày quá hạn cao nhất từng có — signal mạnh nhất về rủi ro
    # max() để bắt được trường hợp xấu nhất dù chỉ xảy ra 1 lần
    pos_dpd_max=("SK_DPD", "max"),
    # Số ngày quá hạn trung bình — phản ánh mức độ quá hạn "thường xuyên"
    # Khác với max (mức độ nặng nhất), mean đo hành vi trả nợ tổng thể
    pos_dpd_mean=("SK_DPD", "mean"),
    # Số ngày quá hạn có tolerance cao nhất (bỏ qua khoản nợ nhỏ)
    # Đáng tin cậy hơn SK_DPD vì loại bỏ noise từ các khoản tiền nhỏ không đáng kể
    pos_dpd_def_max=("SK_DPD_DEF", "max"),
    # Số tháng từng bị quá hạn (SK_DPD > 0)
    # Đo TẦN SUẤT quá hạn — khác với dpd_max (mức độ nặng)
    # Ví dụ: quá hạn 5 lần mỗi lần 1 ngày vs quá hạn 1 lần 30 ngày — rủi ro khác nhau
    pos_dpd_ever_count=("SK_DPD", lambda x: (x > 0).sum()),
    # ===== SỐ KỲ TRẢ =====
    # Trung bình tổng số kỳ trả của các khoản vay POS/Cash
    # Kỳ trả dài hơn → khoản vay lớn hơn hoặc thời hạn dài hơn
    pos_cnt_instalment_mean=("CNT_INSTALMENT", "mean"),
    # Trung bình số kỳ còn lại phải trả
    # Nếu còn nhiều kỳ → gánh nặng nợ trong tương lai còn lớn
    # Kết hợp với CNT_INSTALMENT để tính % đã trả:
    # (CNT_INSTALMENT - CNT_INSTALMENT_FUTURE) / CNT_INSTALMENT
    pos_cnt_instalment_future_mean=("CNT_INSTALMENT_FUTURE", "mean"),
    # ===== TRẠNG THÁI HỢP ĐỒNG =====
    # Flag: khách hàng từng có khoản vay ở trạng thái HIGH RISK không (binary 0/1)
    # High risk = Demand (ngân hàng yêu cầu trả ngay) hoặc Amortized debt (nợ xấu)
    # max() trên cờ 0/1 → 1 nếu TỪNG có bất kỳ tháng nào ở trạng thái này
    pos_high_risk_ever=("POS_HIGHRISK_FLAG", "max"),
    # Tỷ lệ % số tháng ở trạng thái HIGH RISK
    # Bổ sung cho pos_high_risk_ever: biết không chỉ "có hay không" mà còn "nhiều hay ít"
    # Ví dụ: 1/100 tháng high risk vs 50/100 tháng high risk — rủi ro rất khác nhau
    pos_high_risk_rate=("POS_HIGHRISK_FLAG", "mean"),
    # Tỷ lệ % số tháng ở trạng thái COMPLETED (đã hoàn tất trả nợ)
    # Dấu hiệu TÍCH CỰC — tỷ lệ Completed cao → khách hàng có lịch sử trả nợ tốt
    # mean() trên cờ 0/1 → tỷ lệ % tháng Completed trên tổng số tháng
    pos_completed_rate=("POS_COMPLETED_FLAG", "mean"),
)

print(f"Shape pos_cash_agg: {pos_cash_agg.shape}")
print(pos_cash_agg.head())

In [ ]:
print(install_payments.shape)
print(install_payments.isnull().sum())
print(install_payments.describe())

In [ ]:
# ===== BƯỚC 1: CHUYỂN DAYS SANG SỐ DƯƠNG =====
# DAYS_INSTALMENT và DAYS_ENTRY_PAYMENT đều âm (tính ngược từ ngày nộp đơn hiện tại)
# Convert sang dương để dễ tính toán và đọc hiểu hơn
install_payments["DAYS_INSTALMENT"] = install_payments["DAYS_INSTALMENT"].abs()
install_payments["DAYS_ENTRY_PAYMENT"] = install_payments["DAYS_ENTRY_PAYMENT"].abs()

# ===== BƯỚC 2: TẠO FEATURE MỚI =====

# Feature 1: Số ngày trả trễ/sớm
# Sau khi abs(): DAYS_ENTRY_PAYMENT < DAYS_INSTALMENT nghĩa là trả sớm (ngày thực tế gần hơn)
# Dương = trả trễ (nguy hiểm), Âm = trả sớm (tốt), NaN = chưa trả
install_payments["payment_delay"] = (
    install_payments["DAYS_ENTRY_PAYMENT"] - install_payments["DAYS_INSTALMENT"]
)

# Feature 2: Tỷ lệ số tiền thực trả / số tiền phải trả
# = 1.0: trả đúng số tiền quy định
# < 1.0: trả thiếu → rủi ro (ví dụ chỉ trả 80% → 0.8)
# > 1.0: trả dư → tích cực
# replace(0, np.nan): tránh chia cho 0 khi AMT_INSTALMENT = 0
install_payments["payment_ratio"] = install_payments["AMT_PAYMENT"] / install_payments[
    "AMT_INSTALMENT"
].replace(0, np.nan)

# Feature 3: Số tiền thiếu so với quy định (nếu trả thiếu)
# Dương = thiếu bao nhiêu tiền, Âm = trả dư bao nhiêu tiền
# NaN = chưa trả
install_payments["payment_diff"] = (
    install_payments["AMT_INSTALMENT"] - install_payments["AMT_PAYMENT"]
)

# Feature 4: Flag kỳ trả trễ (binary 0/1)
# 1 = kỳ này trả trễ (dù chỉ 1 ngày)
# 0 = trả đúng hạn hoặc sớm
# NaN payment_delay (chưa trả) -> coi là trễ vì chưa thanh toán
install_payments["is_late"] = (install_payments["payment_delay"] > 0).astype(int)

# Kiểm tra các feature vừa tạo
print(
    install_payments[
        ["payment_delay", "payment_ratio", "payment_diff", "is_late"]
    ].describe()
)

In [ ]:
print(install_payments["payment_delay"].quantile([0.001, 0.01, 0.99, 0.999]))
print(install_payments["payment_ratio"].quantile([0.99, 0.999, 0.9999]))
print((install_payments["payment_delay"] == 0).mean())  # tỷ lệ trả đúng ngày

In [ ]:
# ===== XỬ LÝ OUTLIER =====

# --- payment_delay ---
# 99.9% nằm trong [-135, 135] ngày — giá trị ngoài khoảng này là bất thường
# Clip 2 đầu vì:
# - Trả sớm 8 năm (-2884 ngày) là vô lý về mặt thực tế
# - Trả trễ 9 năm (3189 ngày) cũng vô lý — khoản vay không thể kéo dài như vậy
delay_low = install_payments["payment_delay"].quantile(0.001)
delay_high = install_payments["payment_delay"].quantile(0.999)
install_payments["payment_delay"] = install_payments["payment_delay"].clip(
    lower=delay_low, upper=delay_high
)

# --- payment_ratio ---
# Công thức: AMT_PAYMENT / AMT_INSTALMENT
# >= 1.0: trả đủ hoặc dư → tích cực, KHÔNG clip lower
# < 1.0: trả thiếu → rủi ro quan trọng, cần giữ nguyên kể cả gần 0
# Chỉ clip upper vì max = 194,250 là bất thường (có thể do lỗi data)
# 99.9th percentile = 11.2 — đủ rộng để giữ các trường hợp trả dư hợp lý
ratio_cap = install_payments["payment_ratio"].quantile(0.999)
install_payments["payment_ratio"] = install_payments["payment_ratio"].clip(
    upper=ratio_cap
)

# --- payment_diff ---
# AMT_INSTALMENT - AMT_PAYMENT
# Dương = thiếu tiền, Âm = trả dư, NaN = chưa trả
# Clip 2 đầu theo 0.1% và 99.9% để loại outlier cực đoan
diff_low = install_payments["payment_diff"].quantile(0.001)
diff_high = install_payments["payment_diff"].quantile(0.999)
install_payments["payment_diff"] = install_payments["payment_diff"].clip(
    lower=diff_low, upper=diff_high
)

# Kiểm tra lại sau khi clip
print("Sau khi clip:")
print(install_payments[["payment_delay", "payment_ratio", "payment_diff"]].describe())

In [ ]:
install_payments_agg = install_payments.groupby("SK_ID_CURR", as_index=False).agg(
    # ===== ĐỘ DÀI LỊCH SỬ =====
    # Tổng số kỳ thanh toán đã ghi nhận — khách hàng có nhiều kỳ = lịch sử dài hơn
    install_count=("NUM_INSTALMENT_NUMBER", "count"),
    # ===== SỐ NGÀY TRẢ TRỄ/SỚM =====
    # Trung bình số ngày trễ/sớm — phản ánh hành vi trả nợ tổng thể
    # Dương = thường xuyên trễ, Âm = thường xuyên trả sớm
    install_payment_delay_mean=("payment_delay", "mean"),
    # Số ngày trễ cao nhất từng có — bắt được trường hợp xấu nhất
    install_payment_delay_max=("payment_delay", "max"),
    # Số kỳ từng trả trễ — đo TẦN SUẤT trễ, khác với delay_max (mức độ nặng nhất)
    install_late_count=("is_late", "sum"),
    # Tỷ lệ % kỳ trả trễ / tổng số kỳ — chuẩn hóa theo độ dài lịch sử
    # Ví dụ: trễ 10/100 kỳ vs trễ 10/20 kỳ — rủi ro rất khác nhau
    install_late_rate=("is_late", "mean"),
    # ===== SỐ TIỀN =====
    # Trung bình tỷ lệ thực trả / phải trả
    # Gần 1.0 = trả đúng, < 1.0 = thường xuyên trả thiếu -> rủi ro
    install_payment_ratio_mean=("payment_ratio", "mean"),
    # Tỷ lệ thực trả / phải trả thấp nhất từng có
    # Bắt được kỳ trả thiếu nghiêm trọng nhất
    install_payment_ratio_min=("payment_ratio", "min"),
    # Trung bình số tiền thiếu/dư mỗi kỳ
    # Dương = thường thiếu tiền, Âm = thường trả dư
    install_payment_diff_mean=("payment_diff", "mean"),
    # Tổng số tiền thiếu cộng dồn — phản ánh tổng gánh nặng nợ chưa trả
    install_payment_diff_sum=("payment_diff", "sum"),
    # ===== PHIÊN BẢN LỊCH TRẢ NỢ =====
    # Số lần thay đổi lịch trả nợ (gia hạn, tái cơ cấu...)
    # Thay đổi nhiều lần -> khách hàng gặp khó khăn tài chính liên tục
    install_version_max=("NUM_INSTALMENT_VERSION", "max"),
)

print(f"Shape install_agg: {install_payments_agg.shape}")
print(install_payments_agg.head())

In [ ]:
print(install_payments["NUM_INSTALMENT_VERSION"].value_counts().head(10))
print(install_payments_agg["install_version_max"].describe())
print(install_payments_agg["install_version_max"].quantile([0.95, 0.99, 0.999, 0.9999]))
print((install_payments_agg["install_version_max"] > 10).sum())

In [ ]:
version_cap = install_payments_agg["install_version_max"].quantile(0.999)
install_payments_agg["install_version_max"] = install_payments_agg[
    "install_version_max"
].clip(upper=version_cap)

# Tính lại từ install_payments (cấp dòng) rồi merge vào install_agg
cc_rate = install_payments.groupby("SK_ID_CURR", as_index=False).agg(
    # Tỷ lệ % kỳ trả là thẻ tín dụng (version=0)
    # Khách hàng có tỷ lệ cao -> chủ yếu dùng thẻ tín dụng, không phải khoản vay cố định
    install_credit_card_rate=("NUM_INSTALMENT_VERSION", lambda x: (x == 0).mean())
)

install_payments_agg = install_payments_agg.merge(cc_rate, on="SK_ID_CURR", how="left")

print(f"Shape install_payments_agg sau khi thêm feature: {install_payments_agg.shape}")

# Tóm tắt xử lý bảng phụ: card_bal, pos_cash, installments_payments

## Tổng quan sau merge

| | Trước merge | Sau merge |
|---|---|---|
| app_train | (307,511, 194) | (307,511, 228) |
| app_test | (48,744, 187) | (48,744, 221) |

---

## 1. credit_card_balance

### Đặc điểm
- 3,840,312 dòng, 23 cột
- Mỗi dòng = 1 tháng lịch sử của 1 thẻ tín dụng
- Join key: `SK_ID_CURR`
- **103,558 khách hàng unique** (~33.7% tổng app_train)
- Missing sau merge: **~71.7%** — hợp lý vì không phải ai cũng có thẻ tín dụng

### Xử lý đặc biệt
- Impute 0 cho các cột giao dịch (~750k missing): `AMT_DRAWINGS_ATM/OTHER/POS_CURRENT`, `CNT_DRAWINGS_*`, `AMT_PAYMENT_CURRENT` — tháng không có giao dịch thì = 0
- Giữ NaN cho `AMT_INST_MIN_REGULARITY`, `CNT_INSTALMENT_MATURE_CUM` (~305k missing) — ý nghĩa khác, không rõ là 0 hay thiếu thông tin
- Clip `SK_DPD` theo 99.9% (1,492) và `SK_DPD_DEF` theo 99.9% (8)
- Tạo `utilization = AMT_BALANCE / AMT_CREDIT_LIMIT_ACTUAL` → clip [-2, 2]
- Giá trị âm của `utilization` (trả dư vào thẻ) và >1 (vượt hạn mức) được giữ trong [-2, 2] vì có ý nghĩa thực tế

### Features tạo ra (aggregate theo SK_ID_CURR)

| Feature | Ý nghĩa | Hàm |
|---|---|---|
| `card_months_count` | Độ dài lịch sử thẻ tín dụng | `count` |
| `card_amt_balance_mean` | Số dư trung bình hàng tháng | `mean` |
| `card_amt_balance_max` | Số dư cao nhất từng có | `max` |
| `card_amt_credit_limit_mean` | Hạn mức tín dụng trung bình | `mean` |
| `card_amt_drawings_mean` | Số tiền giao dịch trung bình | `mean` |
| `card_amt_payment_mean` | Số tiền thanh toán trung bình | `mean` |
| `card_amt_payment_total_mean` | Tổng tiền thanh toán trung bình | `mean` |
| `card_dpd_max` | Số ngày quá hạn cao nhất | `max` |
| `card_dpd_mean` | Số ngày quá hạn trung bình | `mean` |
| `card_dpd_def_max` | Số ngày quá hạn (có tolerance) cao nhất | `max` |
| `card_dpd_ever_count` | Số tháng từng quá hạn | `sum` |
| `card_utilization_mean` | Tỷ lệ sử dụng hạn mức trung bình | `mean` |
| `card_utilization_max` | Tỷ lệ sử dụng hạn mức cao nhất | `max` |
| `card_missing` | Flag: không có thẻ tín dụng | binary 0/1 |

---

## 2. POS_CASH_balance

### Đặc điểm
- 10,001,358 dòng, 8 cột
- Mỗi dòng = 1 tháng lịch sử của 1 khoản vay POS/Cash
- Join key: `SK_ID_CURR`
- **337,252 khách hàng unique** (~109.7% — có khách hàng trong app_test không có trong app_train)
- Missing sau merge: **~5.9%** — rất thấp, hầu hết khách hàng có lịch sử POS/Cash

### Xử lý đặc biệt
- Clip `SK_DPD` theo 99.9% (2,214) và `SK_DPD_DEF` theo 99.9% (19)
- Nhóm `NAME_CONTRACT_STATUS` thành 5 nhóm:

| Nhóm | Trạng thái gốc | Ý nghĩa |
|---|---|---|
| `Active` | Active | Đang hoạt động |
| `Completed` | Completed | Đã hoàn tất |
| `Pending` | Signed, Approved, XNA | Đã duyệt/ký, chờ xử lý |
| `High_risk` | Demand, Amortized debt | Rủi ro cao |
| `Canceled` | Returned to the store, Canceled | Đã hủy |

- Tạo flag nhị phân `pos_High_risk` và `pos_Completed` từ nhóm trên

### Features tạo ra (aggregate theo SK_ID_CURR)

| Feature | Ý nghĩa | Hàm |
|---|---|---|
| `pos_months_count` | Độ dài lịch sử POS/Cash | `count` |
| `pos_dpd_max` | Số ngày quá hạn cao nhất | `max` |
| `pos_dpd_mean` | Số ngày quá hạn trung bình | `mean` |
| `pos_dpd_def_max` | Số ngày quá hạn (có tolerance) cao nhất | `max` |
| `pos_dpd_ever_count` | Số tháng từng quá hạn | `sum` |
| `pos_cnt_instalment_mean` | Trung bình tổng số kỳ trả | `mean` |
| `pos_cnt_instalment_future_mean` | Trung bình số kỳ còn lại | `mean` |
| `pos_high_risk_ever` | Từng có trạng thái High_risk chưa | `max` |
| `pos_high_risk_rate` | Tỷ lệ % tháng ở trạng thái High_risk | `mean` |
| `pos_completed_rate` | Tỷ lệ % tháng ở trạng thái Completed | `mean` |

---

## 3. installments_payments

### Đặc điểm
- 13,605,401 dòng, 8 cột
- Mỗi dòng = 1 kỳ thanh toán của 1 khoản vay trước đây
- Join key: `SK_ID_CURR`
- **339,587 khách hàng unique**
- Missing sau merge: **~5.2%**
- Bảng duy nhất cho phép đo hành vi trả nợ thực tế (trả đúng hạn, đủ tiền không)

### Xử lý đặc biệt
- Convert `DAYS_INSTALMENT` và `DAYS_ENTRY_PAYMENT` sang số dương
- Tạo 4 feature mới từ các cột gốc:

| Feature mới | Công thức | Ý nghĩa |
|---|---|---|
| `payment_delay` | `DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT` | Số ngày trễ (+) hoặc sớm (-) |
| `payment_ratio` | `AMT_PAYMENT / AMT_INSTALMENT` | Tỷ lệ thực trả / phải trả |
| `payment_diff` | `AMT_INSTALMENT - AMT_PAYMENT` | Số tiền thiếu (+) hoặc dư (-) |
| `is_late` | `payment_delay > 0` | Flag kỳ trả trễ |

- Clip outlier:
  - `payment_delay`: [-80, 135] ngày (0.1% - 99.9%)
  - `payment_ratio`: upper = 11.2 (99.9%) — không clip lower vì <1 là signal rủi ro
  - `payment_diff`: [-209,370, 51,983] (0.1% - 99.9%)
- Clip `install_version_max` theo 99.9% (22)

### Features tạo ra (aggregate theo SK_ID_CURR)

| Feature | Ý nghĩa | Hàm |
|---|---|---|
| `install_count` | Tổng số kỳ thanh toán | `count` |
| `install_payment_delay_mean` | Trung bình số ngày trễ/sớm | `mean` |
| `install_payment_delay_max` | Số ngày trễ cao nhất | `max` |
| `install_late_count` | Số kỳ từng trả trễ | `sum` |
| `install_late_rate` | Tỷ lệ % kỳ trả trễ | `mean` |
| `install_payment_ratio_mean` | Trung bình tỷ lệ thực trả/phải trả | `mean` |
| `install_payment_ratio_min` | Tỷ lệ thực trả/phải trả thấp nhất | `min` |
| `install_payment_diff_mean` | Trung bình số tiền thiếu/dư | `mean` |
| `install_payment_diff_sum` | Tổng số tiền thiếu/dư | `sum` |
| `install_version_max` | Số lần thay đổi lịch trả nợ cao nhất | `max` |
| `install_credit_card_rate` | Tỷ lệ % kỳ trả là thẻ tín dụng | `mean` |

In [ ]:
# Merge tất cả các bảng phụ vào app_train và app_test
# how='left' để giữ nguyên tất cả khách hàng trong app_train/app_test
# Khách hàng không có trong bảng phụ -> NaN ở các cột tương ứng

app_train = app_train.merge(card_bal_agg, on="SK_ID_CURR", how="left")
app_train = app_train.merge(pos_cash_agg, on="SK_ID_CURR", how="left")
app_train = app_train.merge(install_payments_agg, on="SK_ID_CURR", how="left")

app_test = app_test.merge(card_bal_agg, on="SK_ID_CURR", how="left")
app_test = app_test.merge(pos_cash_agg, on="SK_ID_CURR", how="left")
app_test = app_test.merge(install_payments_agg, on="SK_ID_CURR", how="left")

print(f"app_train shape: {app_train.shape}")
print(f"app_test shape: {app_test.shape}")

# Kiểm tra tỷ lệ missing của các cột mới merge vào
new_cols_card = [col for col in card_bal_agg.columns if col != "SK_ID_CURR"]
new_cols_pos = [col for col in pos_cash_agg.columns if col != "SK_ID_CURR"]
new_cols_install = [col for col in install_payments_agg.columns if col != "SK_ID_CURR"]

print("\nTỷ lệ missing cột card_bal:")
print(app_train[new_cols_card].isnull().mean().round(4))
print("\nTỷ lệ missing cột pos_cash:")
print(app_train[new_cols_pos].isnull().mean().round(4))
print("\nTỷ lệ missing cột install_payments:")
print(app_train[new_cols_install].isnull().mean().round(4))

In [ ]:
app_train['card_missing'] = app_train['card_months_count'].isnull().astype(int)
app_test['card_missing'] = app_test['card_months_count'].isnull().astype(int)

In [ ]:
app_train.to_csv('../Data/app_train_merged.csv', index=False)
app_test.to_csv('../Data/app_test_merged.csv', index=False)

print(f"Đã xuất app_train_merged.csv: {app_train.shape}")
print(f"Đã xuất app_test_merged.csv: {app_test.shape}")